What's Different Between Activities?

Cycling

Frequency: 1.0-1.5 Hz (60-90 RPM pedaling)

Acceleration: Low variance, smooth (0.5-2.0 m/s²)

Gyroscope: Moderate rotation (0.1-1.5 rad/s)

Pattern: Highly periodic

Vertical movement: Low

Orientation: Mostly stable (upright position)


Walking

Frequency: 1.5-2.5 Hz (normal pace)

Acceleration: Medium variance (2.0-8.0 m/s²)

Gyroscope: Moderate rotation (0.5-2.0 rad/s)

Pattern: Periodic (heel strikes)

Vertical movement: Medium

Orientation: Oscillating (phone in pocket)


Running

Frequency: 2.0-3.5 Hz (faster pace)

Acceleration: High variance (5.0-15.0 m/s²)

Gyroscope: Chaotic rotation (1.0-4.0 rad/s)

Pattern: Less periodic, more chaotic

Vertical movement: High (bouncing)

Orientation: Variable


Idle/Stationary


Frequency: ~0 Hz (minimal noise only)

Acceleration: Very low, ~9.8 m/s² (gravity only)

Gyroscope: Near zero (0.0-0.2 rad/s)

Pattern: None (random noise)

Vertical movement: Minimal

Orientation: Very stable


Stairs (Up/Down)


Frequency: 0.8-2.0 Hz (slower than walking)

Acceleration: High variance (3.0-10.0 m/s²)

Gyroscope: Moderate (0.5-2.5 rad/s)

Pattern: Asymmetric (up vs down)

Vertical movement: Very high

Orientation: Changing


##SENSOR DATA ANALYSIS PIPELINE - PHASE 1: VALIDATION

In [ ]:
!apt-get update -y
!apt-get install python3.11 python3.11-distutils
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1
!update-alternatives --config python3

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 https://cli.github.com/packages stable InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,528 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [5,976 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/univers

In [ ]:
!apt-get install python3-pip
!python3 -m pip install --upgrade pip --user

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  javascript-common libjs-sphinxdoc libjs-underscore python3-dev
  python3-pkg-resources python3-setuptools python3-wheel python3.10-dev
Suggested packages:
  apache2 | lighttpd | httpd python-setuptools-doc
The following NEW packages will be installed:
  javascript-common libjs-sphinxdoc libjs-underscore python3-dev python3-pip
  python3-setuptools python3-wheel python3.10-dev
The following packages will be upgraded:
  python3-pkg-resources
1 upgraded, 8 newly installed, 0 to remove and 42 not upgraded.
Need to get 2,815 kB of archives.
After this operation, 10.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 javascript-common all 11+nmu1 [5,936 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libjs-underscore all 1.13.2~dfsg-2 [118 kB]
Get:3 http://archive.ubuntu.com/ubuntu ja

In [ ]:
!python --version

Python 3.10.12


In [ ]:
"""
Sensor Data Analysis Pipeline for Anti-Emulation Detection
Supports multiple activities: cycling, walking, running, idle, stairs, etc.

Author: Anti-Emulation Research Team
Date: 2025-11-10
"""

import os
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import json
from scipy import stats, signal
from scipy.fft import fft, fftfreq
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 8)


class ActivityConfig:
    """Configuration class for activity-specific parameters"""

    CONFIGS = {
        'cycling': {
            'description': 'Cycling activity',
            'expected_frequency_range': (1.0, 1.5),  # Hz (60-90 RPM)
            'expected_acceleration_range': (0.5, 2.0),  # m/s² (smooth)
            'expected_gyroscope_range': (0.1, 1.5),  # rad/s (moderate rotation)
            'periodicity': 'high',  # High periodic patterns
            'vertical_variance': 'low',  # Low vertical movement
            'orientation_stability': 'high',  # Mostly stable orientation
        },
        'walking': {
            'description': 'Walking activity',
            'expected_frequency_range': (1.5, 2.5),  # Hz (normal walking pace)
            'expected_acceleration_range': (2.0, 8.0),  # m/s²
            'expected_gyroscope_range': (0.5, 2.0),  # rad/s
            'periodicity': 'high',
            'vertical_variance': 'medium',
            'orientation_stability': 'medium',
        },
        'running': {
            'description': 'Running activity',
            'expected_frequency_range': (2.0, 3.5),  # Hz (faster pace)
            'expected_acceleration_range': (5.0, 15.0),  # m/s² (high impact)
            'expected_gyroscope_range': (1.0, 4.0),  # rad/s (more chaotic)
            'periodicity': 'medium',
            'vertical_variance': 'high',
            'orientation_stability': 'low',
        },
        'idle': {
            'description': 'Stationary/Idle',
            'expected_frequency_range': (0.0, 0.5),  # Hz (minimal movement)
            'expected_acceleration_range': (0.0, 0.5),  # m/s² (only noise)
            'expected_gyroscope_range': (0.0, 0.2),  # rad/s (minimal rotation)
            'periodicity': 'none',
            'vertical_variance': 'very_low',
            'orientation_stability': 'very_high',
        },
        'upstairs': {
            'description': 'Climbing stairs',
            'expected_frequency_range': (0.8, 1.5),  # Hz (slower than walking)
            'expected_acceleration_range': (3.0, 10.0),  # m/s²
            'expected_gyroscope_range': (0.5, 2.5),  # rad/s
            'periodicity': 'medium',
            'vertical_variance': 'high',
            'orientation_stability': 'medium',
        },
        'downstairs': {
            'description': 'Descending stairs',
            'expected_frequency_range': (1.0, 2.0),  # Hz
            'expected_acceleration_range': (3.0, 10.0),  # m/s²
            'expected_gyroscope_range': (0.5, 2.5),  # rad/s
            'periodicity': 'medium',
            'vertical_variance': 'high',
            'orientation_stability': 'medium',
        }
    }

    @classmethod
    def get_config(cls, activity: str) -> dict:
        """Get configuration for specific activity"""
        if activity.lower() not in cls.CONFIGS:
            print(f"⚠️  Warning: Activity '{activity}' not found. Using default (cycling)")
            return cls.CONFIGS['cycling']
        return cls.CONFIGS[activity.lower()]

    @classmethod
    def list_activities(cls) -> List[str]:
        """List all supported activities"""
        return list(cls.CONFIGS.keys())


class SensorDataLoader:
    """Load and parse sensor data from ZIP files"""

    def __init__(self, zip_path: str, extract_dir: str = "/home/claude/sensor_data"):
        self.zip_path = zip_path
        self.extract_dir = extract_dir
        self.data = {}
        self.metadata = {}

    def extract_zip(self):
        """Extract ZIP file contents"""
        print(f"\n{'='*60}")
        print(f"📦 Extracting ZIP file: {Path(self.zip_path).name}")
        print(f"{'='*60}")

        Path(self.extract_dir).mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(self.zip_path, 'r') as zip_ref:
            zip_ref.extractall(self.extract_dir)

        print(f"✅ Extracted to: {self.extract_dir}\n")

    def load_all_sensors(self) -> Dict[str, pd.DataFrame]:
        """Load all CSV files from extracted directory"""
        print(f"{'='*60}")
        print(f"📊 LOADING SENSOR DATA")
        print(f"{'='*60}\n")

        csv_files = list(Path(self.extract_dir).rglob("*.csv"))

        if not csv_files:
            print(f"❌ No CSV files found in {self.extract_dir}")
            return {}

        loaded_count = 0
        skipped_count = 0

        for csv_file in csv_files:
            sensor_name = csv_file.stem  # Filename without extension

            try:
                df = pd.read_csv(csv_file)

                if df.empty or len(df) == 0:
                    print(f"⚠️  Skipping empty: {csv_file.name}")
                    skipped_count += 1
                    continue

                self.data[sensor_name] = df
                print(f"✅ Loaded {csv_file.name:<30} shape={df.shape}")
                loaded_count += 1

            except Exception as e:
                print(f"❌ Error loading {csv_file.name}: {str(e)}")
                skipped_count += 1

        print(f"\n{'='*60}")
        print(f"📈 Summary: {loaded_count} loaded, {skipped_count} skipped")
        print(f"{'='*60}\n")

        return self.data

    def get_sensor(self, sensor_name: str) -> Optional[pd.DataFrame]:
        """Get specific sensor data"""
        return self.data.get(sensor_name, None)

    def list_sensors(self) -> List[str]:
        """List all loaded sensors"""
        return list(self.data.keys())


class SensorDataValidator:
    """Validate sensor data quality and physical consistency"""

    def __init__(self, data: Dict[str, pd.DataFrame], activity: str = 'cycling'):
        self.data = data
        self.activity = activity
        self.config = ActivityConfig.get_config(activity)
        self.validation_results = {}

    def validate_all(self) -> Dict:
        """Run all validation checks"""
        print(f"\n{'='*60}")
        print(f"🔍 VALIDATION FOR ACTIVITY: {self.activity.upper()}")
        print(f"{'='*60}\n")

        results = {
            'basic_checks': self._check_basic_quality(),
            'physical_constraints': self._check_physical_constraints(),
            'temporal_consistency': self._check_temporal_consistency(),
            'cross_sensor_validation': self._check_cross_sensor_consistency(),
            'activity_specific': self._check_activity_patterns(),
        }

        self.validation_results = results
        self._print_summary()

        return results

    def _check_basic_quality(self) -> Dict:
        """Check for missing values, duplicates, outliers"""
        print("📋 Basic Quality Checks:")
        results = {}

        for sensor_name, df in self.data.items():
            sensor_results = {
                'missing_values': df.isnull().sum().sum(),
                'duplicate_rows': df.duplicated().sum(),
                'total_rows': len(df),
                'columns': list(df.columns),
            }

            # Check for outliers using IQR method
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            outlier_counts = {}

            for col in numeric_cols:
                if col.lower() != 'time':  # Skip timestamp columns
                    Q1 = df[col].quantile(0.25)
                    Q3 = df[col].quantile(0.75)
                    IQR = Q3 - Q1
                    outliers = ((df[col] < (Q1 - 3 * IQR)) | (df[col] > (Q3 + 3 * IQR))).sum()
                    outlier_counts[col] = outliers

            sensor_results['outliers'] = outlier_counts
            results[sensor_name] = sensor_results

            # Print summary
            status = "✅" if sensor_results['missing_values'] == 0 else "⚠️"
            print(f"  {status} {sensor_name}: {sensor_results['total_rows']} rows, "
                  f"{sensor_results['missing_values']} missing, "
                  f"{sensor_results['duplicate_rows']} duplicates")

        print()
        return results

    def _check_physical_constraints(self) -> Dict:
        """Check if sensor values are physically plausible"""
        print("⚙️  Physical Constraint Checks:")
        results = {}

        # Check accelerometer gravity constraint
        if 'Accelerometer' in self.data:
            acc = self.data['Accelerometer']
            if all(col in acc.columns for col in ['x', 'y', 'z']):
                magnitude = np.sqrt(acc['x']**2 + acc['y']**2 + acc['z']**2)
                mean_mag = magnitude.mean()

                # For idle, should be close to 9.8 m/s²
                # For movement, will vary but should be reasonable
                if self.activity == 'idle':
                    is_valid = 9.0 < mean_mag < 10.5
                    print(f"  {'✅' if is_valid else '⚠️'} Accelerometer magnitude (idle): "
                          f"{mean_mag:.2f} m/s² (expected ~9.8)")
                else:
                    is_valid = 5.0 < mean_mag < 25.0  # Broader range for movement
                    print(f"  {'✅' if is_valid else '⚠️'} Accelerometer magnitude: "
                          f"{mean_mag:.2f} m/s² (expected 5-25)")

                results['accelerometer_gravity'] = {
                    'mean_magnitude': mean_mag,
                    'valid': is_valid
                }

        # Check gyroscope reasonable range
        if 'Gyroscope' in self.data:
            gyro = self.data['Gyroscope']
            if all(col in gyro.columns for col in ['x', 'y', 'z']):
                max_rotation = gyro[['x', 'y', 'z']].abs().max().max()

                # Most human movements < 10 rad/s
                is_valid = max_rotation < 10.0
                print(f"  {'✅' if is_valid else '⚠️'} Gyroscope max rotation: "
                      f"{max_rotation:.2f} rad/s (should be < 10)")

                results['gyroscope_range'] = {
                    'max_rotation': max_rotation,
                    'valid': is_valid
                }

        # Check magnetometer reasonable range
        if 'Magnetometer' in self.data:
            mag = self.data['Magnetometer']
            if all(col in mag.columns for col in ['x', 'y', 'z']):
                magnitude = np.sqrt(mag['x']**2 + mag['y']**2 + mag['z']**2)
                mean_mag = magnitude.mean()

                # Earth's magnetic field is typically 25-65 μT
                is_valid = 20 < mean_mag < 100
                print(f"  {'✅' if is_valid else '⚠️'} Magnetometer magnitude: "
                      f"{mean_mag:.2f} μT (expected 25-65)")

                results['magnetometer_range'] = {
                    'mean_magnitude': mean_mag,
                    'valid': is_valid
                }

        print()
        return results

    def _check_temporal_consistency(self) -> Dict:
        """Check timestamp consistency and sampling rates"""
        print("⏱️  Temporal Consistency Checks:")
        results = {}

        for sensor_name, df in self.data.items():
            # Find timestamp column
            time_col = None
            for col in df.columns:
                if 'time' in col.lower():
                    time_col = col
                    break

            if time_col:
                time_diffs = df[time_col].diff().dropna()

                # Convert to seconds if in milliseconds/nanoseconds
                if time_diffs.median() > 1000:
                    time_diffs = time_diffs / 1000.0  # Assume milliseconds

                sampling_rate = 1.0 / time_diffs.median() if time_diffs.median() > 0 else 0

                results[sensor_name] = {
                    'sampling_rate_hz': sampling_rate,
                    'mean_interval_ms': time_diffs.mean() * 1000,
                    'std_interval_ms': time_diffs.std() * 1000,
                }

                print(f"  ✅ {sensor_name}: ~{sampling_rate:.1f} Hz "
                      f"(interval: {time_diffs.mean()*1000:.1f}±{time_diffs.std()*1000:.1f} ms)")

        print()
        return results

    def _check_cross_sensor_consistency(self) -> Dict:
        """Check if sensors are synchronized and correlated"""
        print("🔗 Cross-Sensor Consistency:")
        results = {}

        # Check if accelerometer and gyroscope are correlated
        if 'Accelerometer' in self.data and 'Gyroscope' in self.data:
            acc = self.data['Accelerometer']
            gyro = self.data['Gyroscope']

            # Align by length
            min_len = min(len(acc), len(gyro))

            if all(col in acc.columns for col in ['x', 'y', 'z']) and \
               all(col in gyro.columns for col in ['x', 'y', 'z']):

                correlations = []
                for axis in ['x', 'y', 'z']:
                    corr = np.corrcoef(acc[axis][:min_len], gyro[axis][:min_len])[0, 1]
                    correlations.append(corr)

                mean_corr = np.mean(np.abs(correlations))
                print(f"  ✅ Accelerometer-Gyroscope correlation: {mean_corr:.3f}")

                results['acc_gyro_correlation'] = mean_corr

        print()
        return results

    def _check_activity_patterns(self) -> Dict:
        """Check if data matches expected patterns for this activity"""
        print(f"🎯 Activity-Specific Pattern Checks ({self.activity}):")
        results = {}

        # Check frequency characteristics
        if 'Accelerometer' in self.data:
            acc = self.data['Accelerometer']
            if 'x' in acc.columns:
                # Perform FFT to find dominant frequency
                signal_data = acc['x'].values
                fft_vals = fft(signal_data)
                freqs = fftfreq(len(signal_data), d=0.02)  # Assuming ~50Hz sampling

                # Get positive frequencies only
                pos_mask = freqs > 0
                pos_freqs = freqs[pos_mask]
                pos_fft = np.abs(fft_vals[pos_mask])

                # Find dominant frequency
                dominant_idx = np.argmax(pos_fft)
                dominant_freq = pos_freqs[dominant_idx]

                expected_range = self.config['expected_frequency_range']
                is_valid = expected_range[0] <= dominant_freq <= expected_range[1]

                print(f"  {'✅' if is_valid else '⚠️'} Dominant frequency: {dominant_freq:.2f} Hz "
                      f"(expected: {expected_range[0]}-{expected_range[1]} Hz)")

                results['dominant_frequency'] = {
                    'value': dominant_freq,
                    'expected_range': expected_range,
                    'valid': is_valid
                }

        # Check acceleration magnitude matches activity
        if 'Accelerometer' in self.data:
            acc = self.data['Accelerometer']
            if all(col in acc.columns for col in ['x', 'y', 'z']):
                # Calculate variance in acceleration (indicator of movement intensity)
                acc_variance = acc[['x', 'y', 'z']].var().mean()

                expected_range = self.config['expected_acceleration_range']

                # For variance check, we use the square of acceleration range
                expected_var_range = (expected_range[0]**2, expected_range[1]**2)
                is_valid = expected_var_range[0] <= acc_variance <= expected_var_range[1]

                print(f"  {'✅' if is_valid else '⚠️'} Acceleration variance: {acc_variance:.2f} "
                      f"(expected: {expected_var_range[0]:.1f}-{expected_var_range[1]:.1f})")

                results['acceleration_variance'] = {
                    'value': acc_variance,
                    'expected_range': expected_var_range,
                    'valid': is_valid
                }

        print()
        return results

    def _print_summary(self):
        """Print validation summary"""
        print(f"{'='*60}")
        print(f"📊 VALIDATION SUMMARY")
        print(f"{'='*60}\n")

        # Count issues
        issues = []

        # Check basic quality
        for sensor, data in self.validation_results['basic_checks'].items():
            if data['missing_values'] > 0:
                issues.append(f"Missing values in {sensor}")
            if data['duplicate_rows'] > 0:
                issues.append(f"Duplicate rows in {sensor}")

        # Check physical constraints
        for check, data in self.validation_results['physical_constraints'].items():
            if not data.get('valid', True):
                issues.append(f"Physical constraint failed: {check}")

        # Check activity patterns
        for check, data in self.validation_results['activity_specific'].items():
            if isinstance(data, dict) and not data.get('valid', True):
                issues.append(f"Activity pattern mismatch: {check}")

        if not issues:
            print("✅ All validation checks passed!")
        else:
            print(f"⚠️  Found {len(issues)} issue(s):")
            for issue in issues:
                print(f"   - {issue}")

        print(f"\n{'='*60}\n")


def main():
    """Main execution function"""
    print("\n" + "="*60)
    print("SENSOR DATA ANALYSIS PIPELINE - PHASE 1: VALIDATION")
    print("="*60)

    # List available activities
    print("\n📌 Supported Activities:")
    for activity in ActivityConfig.list_activities():
        config = ActivityConfig.get_config(activity)
        print(f"   • {activity}: {config['description']}")

    print("\n" + "="*60)
    print("INSTRUCTIONS:")
    print("="*60)
    print("1. Place your ZIP file in /mnt/user-data/uploads/")
    print("2. The pipeline will extract and analyze all sensors")
    print("3. Validation will use activity-specific thresholds")
    print("4. Results will be saved for further analysis")
    print("="*60 + "\n")


if __name__ == "__main__":
    main()


SENSOR DATA ANALYSIS PIPELINE - PHASE 1: VALIDATION

📌 Supported Activities:
   • cycling: Cycling activity
   • walking: Walking activity
   • running: Running activity
   • idle: Stationary/Idle
   • upstairs: Climbing stairs
   • downstairs: Descending stairs

INSTRUCTIONS:
1. Place your ZIP file in /mnt/user-data/uploads/
2. The pipeline will extract and analyze all sensors
3. Validation will use activity-specific thresholds
4. Results will be saved for further analysis



#Cycling analysis.py

In [ ]:
"""
Example Usage: Analyzing Cycling Sensor Data
This script demonstrates how to use the pipeline for cycling activity
"""

# import sys
# sys.path.append('/home/claude')
# from sensor_data_pipeline import SensorDataLoader, SensorDataValidator, ActivityConfig
import matplotlib.pyplot as plt
import numpy as np
import json

def analyze_cycling_data(zip_path: str):
    """
    Complete analysis pipeline for cycling data

    Args:
        zip_path: Path to ZIP file containing sensor CSVs
    """

    # Step 1: Load the data
    print("\n" + "="*60)
    print("STEP 1: DATA LOADING")
    print("="*60)

    loader = SensorDataLoader(zip_path)
    loader.extract_zip()
    data = loader.load_all_sensors()

    if not data:
        print("❌ No data loaded. Please check your ZIP file.")
        return None

    # Step 2: Validate the data
    print("\n" + "="*60)
    print("STEP 2: DATA VALIDATION")
    print("="*60)

    validator = SensorDataValidator(data, activity='cycling')
    validation_results = validator.validate_all()

    # Step 3: Save validation report
    print("\n" + "="*60)
    print("STEP 3: SAVING RESULTS")
    print("="*60)

    report_path = "/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/cycling_validation_report.json"

    # Convert validation results to JSON-serializable format
    json_results = {}
    for key, value in validation_results.items():
        json_results[key] = convert_to_serializable(value)

    with open(report_path, 'w') as f:
        json.dump(json_results, f, indent=2)

    print(f"✅ Validation report saved to: {report_path}")

    # Step 4: Create visualizations
    print("\n" + "="*60)
    print("STEP 4: CREATING VISUALIZATIONS")
    print("="*60)

    create_basic_visualizations(data, activity='cycling')

    return {
        'data': data,
        'validation': validation_results,
        'loader': loader,
        'validator': validator
    }


def convert_to_serializable(obj):
    """Convert numpy/pandas types to JSON-serializable types"""
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (bool, np.bool_)): # Handle boolean types
        return bool(obj)
    else:
        return obj


def create_basic_visualizations(data: dict, activity: str):
    """Create basic visualizations of sensor data"""

    viz_dir = "/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/visualizations"
    import os
    os.makedirs(viz_dir, exist_ok=True)

    # 1. Accelerometer time series
    if 'Accelerometer' in data:
        acc = data['Accelerometer']
        if all(col in acc.columns for col in ['x', 'y', 'z']):
            plt.figure(figsize=(15, 5))

            time_col = [col for col in acc.columns if 'time' in col.lower()]
            if time_col:
                time = acc[time_col[0]].values
                # Normalize time to start from 0
                time = (time - time[0]) / 1000.0  # Convert to seconds
            else:
                time = np.arange(len(acc))

            plt.plot(time[:5000], acc['x'][:5000], label='X-axis', alpha=0.7)
            plt.plot(time[:5000], acc['y'][:5000], label='Y-axis', alpha=0.7)
            plt.plot(time[:5000], acc['z'][:5000], label='Z-axis', alpha=0.7)

            plt.title(f'Accelerometer Data - {activity.capitalize()}', fontsize=14, fontweight='bold')
            plt.xlabel('Time (seconds)', fontsize=12)
            plt.ylabel('Acceleration (m/s²)', fontsize=12)
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.tight_layout()

            save_path = f"{viz_dir}/accelerometer_{activity}.png"
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"✅ Saved: {save_path}")
            plt.close()

    # 2. Gyroscope time series
    if 'Gyroscope' in data:
        gyro = data['Gyroscope']
        if all(col in gyro.columns for col in ['x', 'y', 'z']):
            plt.figure(figsize=(15, 5))

            time_col = [col for col in gyro.columns if 'time' in col.lower()]
            if time_col:
                time = gyro[time_col[0]].values
                time = (time - time[0]) / 1000.0
            else:
                time = np.arange(len(gyro))

            plt.plot(time[:5000], gyro['x'][:5000], label='X-axis', alpha=0.7)
            plt.plot(time[:5000], gyro['y'][:5000], label='Y-axis', alpha=0.7)
            plt.plot(time[:5000], gyro['z'][:5000], label='Z-axis', alpha=0.7)

            plt.title(f'Gyroscope Data - {activity.capitalize()}', fontsize=14, fontweight='bold')
            plt.xlabel('Time (seconds)', fontsize=12)
            plt.ylabel('Angular Velocity (rad/s)', fontsize=12)
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.tight_layout()

            save_path = f"{viz_dir}/gyroscope_{activity}.png"
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"✅ Saved: {save_path}")
            plt.close()

    # 3. Multi-sensor correlation heatmap
    if len(data) >= 2:
        create_correlation_heatmap(data, activity, viz_dir)

    print(f"\n📁 All visualizations saved in: {viz_dir}/")


def create_correlation_heatmap(data: dict, activity: str, viz_dir: str):
    """Create correlation heatmap between sensors"""
    import seaborn as sns

    # Collect all numeric columns from main sensors
    sensor_data = []
    column_names = []

    for sensor_name in ['Accelerometer', 'Gyroscope', 'Magnetometer']:
        if sensor_name in data:
            df = data[sensor_name]
            for axis in ['x', 'y', 'z']:
                if axis in df.columns:
                    sensor_data.append(df[axis].values)
                    column_names.append(f"{sensor_name}_{axis}")

    if len(sensor_data) >= 2:
        # Find minimum length
        min_len = min(len(s) for s in sensor_data)

        # Truncate all to same length and create DataFrame
        import pandas as pd
        truncated_data = {name: data[:min_len] for name, data in zip(column_names, sensor_data)}
        combined_df = pd.DataFrame(truncated_data)

        # Compute correlation matrix
        corr_matrix = combined_df.corr()

        # Plot
        plt.figure(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
                    center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
        plt.title(f'Cross-Sensor Correlation Matrix - {activity.capitalize()}',
                  fontsize=14, fontweight='bold')
        plt.tight_layout()

        save_path = f"{viz_dir}/correlation_matrix_{activity}.png"
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✅ Saved: {save_path}")
        plt.close()


# Example usage
if __name__ == "__main__":
    print("\n" + "="*60)
    print("CYCLING DATA ANALYSIS - EXAMPLE USAGE")
    print("="*60 + "\n")

    # Example path - replace with actual path to your ZIP file
    example_zip = "/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/cycling.zip"

    print("📝 To use this script:")
    print(f"   1. Upload your cycling ZIP file to: {example_zip}")
    print("   2. Run: python3 /content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/cycling.zip")
    print("\n" + "="*60 + "\n")

    # Uncomment below to run when you have the file:
    results = analyze_cycling_data(example_zip)


CYCLING DATA ANALYSIS - EXAMPLE USAGE

📝 To use this script:
   1. Upload your cycling ZIP file to: /content/drive/MyDrive/CS5103_IITH_PMA_project/Dataset/cycling.zip
   2. Run: python3 /content/drive/MyDrive/CS5103_IITH_PMA_project/Dataset/cycling.zip



STEP 1: DATA LOADING

📦 Extracting ZIP file: cycling.zip
✅ Extracted to: /home/claude/sensor_data

📊 LOADING SENSOR DATA

✅ Loaded Magnetometer.csv               shape=(23422, 5)
✅ Loaded Battery.csv                    shape=(240, 5)
❌ Error loading WatchGyroscope.csv: No columns to parse from file
❌ Error loading WatchAccelerometer.csv: No columns to parse from file
❌ Error loading Annotation.csv: No columns to parse from file
✅ Loaded Microphone.csv                 shape=(2059, 3)
❌ Error loading Location.csv: No columns to parse from file
✅ Loaded Light.csv                      shape=(2307, 3)
❌ Error loading HeartRate.csv: No columns to parse from file
❌ Error loading WiFi.csv: No columns to parse from file
❌ Error loading WatchB

#Test pipeline.py

In [ ]:
"""
Quick Test Script - Demonstrates Pipeline Functionality
Run this to see how the pipeline works before uploading your data
"""

#import sys
#sys.path.append('/home/claude')
#from sensor_data_pipeline import ActivityConfig, SensorDataValidator
import pandas as pd
import numpy as np

def generate_mock_cycling_data():
    """Generate mock cycling sensor data for demonstration"""

    print("\n" + "="*60)
    print("🧪 GENERATING MOCK CYCLING DATA FOR TESTING")
    print("="*60 + "\n")

    # Simulate 10 seconds of data at 50 Hz
    duration = 10  # seconds
    sample_rate = 50  # Hz
    n_samples = duration * sample_rate

    time = np.arange(n_samples) * (1000 / sample_rate)  # milliseconds

    # Cycling frequency: ~1.2 Hz (72 RPM)
    cycling_freq = 1.2

    # Accelerometer: smooth periodic motion
    acc_x = 0.5 * np.sin(2 * np.pi * cycling_freq * time / 1000) + np.random.normal(0, 0.1, n_samples)
    acc_y = 0.3 * np.cos(2 * np.pi * cycling_freq * time / 1000) + np.random.normal(0, 0.1, n_samples)
    acc_z = 9.8 + 0.2 * np.sin(2 * np.pi * cycling_freq * time / 1000) + np.random.normal(0, 0.15, n_samples)

    accelerometer = pd.DataFrame({
        'time': time,
        'x': acc_x,
        'y': acc_y,
        'z': acc_z
    })

    # Gyroscope: low rotation for cycling
    gyro_x = 0.3 * np.sin(2 * np.pi * cycling_freq * time / 1000) + np.random.normal(0, 0.05, n_samples)
    gyro_y = 0.2 * np.cos(2 * np.pi * cycling_freq * time / 1000) + np.random.normal(0, 0.05, n_samples)
    gyro_z = 0.1 * np.sin(2 * np.pi * cycling_freq * time / 1000) + np.random.normal(0, 0.03, n_samples)

    gyroscope = pd.DataFrame({
        'time': time,
        'x': gyro_x,
        'y': gyro_y,
        'z': gyro_z
    })

    # Magnetometer: stable field
    mag_x = 25 + np.random.normal(0, 2, n_samples)
    mag_y = 30 + np.random.normal(0, 2, n_samples)
    mag_z = 40 + np.random.normal(0, 2, n_samples)

    magnetometer = pd.DataFrame({
        'time': time,
        'x': mag_x,
        'y': mag_y,
        'z': mag_z
    })

    print("✅ Generated mock sensor data:")
    print(f"   - Accelerometer: {accelerometer.shape}")
    print(f"   - Gyroscope: {gyroscope.shape}")
    print(f"   - Magnetometer: {magnetometer.shape}")
    print(f"   - Duration: {duration}s at {sample_rate}Hz\n")

    return {
        'Accelerometer': accelerometer,
        'Gyroscope': gyroscope,
        'Magnetometer': magnetometer
    }


def demonstrate_activity_differences():
    """Show how validation differs between activities"""

    print("\n" + "="*60)
    print("📊 DEMONSTRATION: ACTIVITY-SPECIFIC VALIDATION")
    print("="*60 + "\n")

    # Generate mock data
    data = generate_mock_cycling_data()

    # Test with different activity configurations
    activities = ['cycling']  # , 'walking', 'running', 'idle'

    print("\n" + "="*60)
    print("TESTING SAME DATA WITH DIFFERENT ACTIVITY LABELS")
    print("="*60 + "\n")

    print("This demonstrates how the SAME data is validated")
    print("differently based on activity type:\n")

    for activity in activities:
        print(f"\n{'─'*60}")
        print(f"Testing as: {activity.upper()}")
        print(f"{'─'*60}")

        validator = SensorDataValidator(data, activity=activity)

        # Just show activity-specific checks
        config = ActivityConfig.get_config(activity)
        print(f"Expected frequency: {config['expected_frequency_range']} Hz")
        print(f"Expected acceleration: {config['expected_acceleration_range']} m/s²")
        print(f"Periodicity: {config['periodicity']}")

        # Run validation
        results = validator.validate_all()

        print("\n")


def show_configuration_details():
    """Display all activity configurations"""

    print("\n" + "="*60)
    print("📋 ALL ACTIVITY CONFIGURATIONS")
    print("="*60 + "\n")

    activities = ActivityConfig.list_activities()

    for activity in activities:
        config = ActivityConfig.get_config(activity)
        print(f"\n{'─'*60}")
        print(f"⚙️  {activity.upper()}")
        print(f"{'─'*60}")
        print(f"Description: {config['description']}")
        print(f"Frequency range: {config['expected_frequency_range']} Hz")
        print(f"Acceleration range: {config['expected_acceleration_range']} m/s²")
        print(f"Gyroscope range: {config['expected_gyroscope_range']} rad/s")
        print(f"Periodicity: {config['periodicity']}")
        print(f"Vertical variance: {config['vertical_variance']}")
        print(f"Orientation stability: {config['orientation_stability']}")


def main():
    """Main test function"""

    print("\n" + "="*60)
    print("🧪 SENSOR DATA PIPELINE - DEMONSTRATION MODE")
    print("="*60)

    print("\nThis script demonstrates:")
    print("  1. How the pipeline works with different activities")
    print("  2. Same code, different thresholds")
    print("  3. Activity-specific validation\n")

    # Show all configurations
    show_configuration_details()

    # Demonstrate with mock data
    demonstrate_activity_differences()

    print("\n" + "="*60)
    print("✅ DEMONSTRATION COMPLETE")
    print("="*60)
    print("\nKey Takeaway:")
    print("  ➤ The PIPELINE is the same for all activities")
    print("  ➤ Only the VALIDATION THRESHOLDS change")
    print("  ➤ Simply change: activity='cycling' → activity='walking'")
    print("\nNext step: Upload your real cycling data and run:")
    print("  python3 /home/claude/example_cycling_analysis.py")
    print("="*60 + "\n")


if __name__ == "__main__":
    main()


🧪 SENSOR DATA PIPELINE - DEMONSTRATION MODE

This script demonstrates:
  1. How the pipeline works with different activities
  2. Same code, different thresholds
  3. Activity-specific validation


📋 ALL ACTIVITY CONFIGURATIONS


────────────────────────────────────────────────────────────
⚙️  CYCLING
────────────────────────────────────────────────────────────
Description: Cycling activity
Frequency range: (1.0, 1.5) Hz
Acceleration range: (0.5, 2.0) m/s²
Gyroscope range: (0.1, 1.5) rad/s
Periodicity: high
Vertical variance: low
Orientation stability: high

────────────────────────────────────────────────────────────
⚙️  WALKING
────────────────────────────────────────────────────────────
Description: Walking activity
Frequency range: (1.5, 2.5) Hz
Acceleration range: (2.0, 8.0) m/s²
Gyroscope range: (0.5, 2.0) rad/s
Periodicity: high
Vertical variance: medium
Orientation stability: medium

────────────────────────────────────────────────────────────
⚙️  RUNNING
─────────────────────

Validity Check for un correlated data Battery, Battery Temp, Brighness and Light

In [ ]:
import pandas as pd

# --- Define valid ranges for each sensor ---
VALID_RANGES = {
    'Battery': (0, 100),             # Percentage
    'Battery_temp': (20, 60),        # Celsius
    'Light': (0, 10000),             # Lux (can adjust based on sensor)
    'Brightness': (0, 255),          # Scale (0–255)
}

def check_range_validity(df):
    """
    Check if sensor values are within their valid ranges.
    Prints out-of-range stats for each sensor.
    """
    results = {}

    for sensor, (low, high) in VALID_RANGES.items():
        if sensor not in df.columns:
            print(f"[!] Sensor '{sensor}' not found in DataFrame.")
            continue

        out_of_range = df[(df[sensor] < low) | (df[sensor] > high)]
        total = len(df)
        invalid_count = len(out_of_range)
        valid_rate = (1 - invalid_count / total) * 100

        results[sensor] = {
            'valid_range': (low, high),
            'invalid_count': invalid_count,
            'total_samples': total,
            'valid_rate (%)': round(valid_rate, 2)
        }

        if invalid_count > 0:
            print(f"⚠️ {sensor}: {invalid_count}/{total} values outside range {low}-{high}")
        else:
            print(f"✅ {sensor}: All values within range {low}-{high}")

    return pd.DataFrame(results).T


# --- Example usage ---
if __name__ == "__main__":
    # Example dummy data
    data = {
        'Battery': [90, 85, 110, 50, -5],
        'Battery_temp': [25, 30, 65, 22, 45],
        'Light': [300, 8000, 12000, 0, 450],
        'Brightness': [200, 255, 260, 100, -10]
    }

    df = pd.DataFrame(data)
    report = check_range_validity(df)

    print("\n--- Range Validity Report ---")
    print(report)


⚠️ Battery: 2/5 values outside range 0-100
⚠️ Battery_temp: 1/5 values outside range 20-60
⚠️ Light: 1/5 values outside range 0-10000
⚠️ Brightness: 2/5 values outside range 0-255

--- Range Validity Report ---
             valid_range invalid_count total_samples valid_rate (%)
Battery         (0, 100)             2             5           60.0
Battery_temp    (20, 60)             1             5           80.0
Light         (0, 10000)             1             5           80.0
Brightness      (0, 255)             2             5           60.0


In [ ]:
def create_correlation_heatmap(data: dict, activity: str, viz_dir: str):
    """Create correlation heatmap between sensors (using magnitude only)"""
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt
    import numpy as np
    import os

    # List of sensors to check
    sensors = ['Accelerometer', 'Gyroscope', 'Magnetometer']
    magnitude_data = {}

    # Compute magnitude √(x² + y² + z²) for each available sensor
    for sensor_name in sensors:
        if sensor_name in data:
            df = data[sensor_name]

            # Check that x, y, z columns exist
            if all(axis in df.columns for axis in ['x', 'y', 'z']):
                mag = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)
                magnitude_data[sensor_name] = mag.values
            else:
                print(f"[!] Skipping {sensor_name} — missing one of x/y/z")

    if len(magnitude_data) >= 2:
        # Find minimum length among sensors
        min_len = min(len(v) for v in magnitude_data.values())

        # Truncate all to same length
        truncated = {name: vals[:min_len] for name, vals in magnitude_data.items()}

        # Create DataFrame
        combined_df = pd.DataFrame(truncated)

        # Compute correlation matrix
        corr_matrix = combined_df.corr()

        # Plot
        plt.figure(figsize=(8, 6))
        sns.heatmap(
            corr_matrix,
            annot=True,
            fmt='.2f',
            cmap='coolwarm',
            center=0,
            square=True,
            linewidths=1,
            cbar_kws={"shrink": 0.8}
        )
        plt.title(f'Cross-Sensor Magnitude Correlation - {activity.capitalize()}',
                  fontsize=14, fontweight='bold')
        plt.tight_layout()

        # Ensure save directory exists
        os.makedirs(viz_dir, exist_ok=True)
        save_path = f"{viz_dir}/correlation_matrix_{activity}_magnitude.png"
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"✅ Saved correlation heatmap: {save_path}")
    else:
        print("⚠️ Not enough sensors with magnitude data to compute correlation.")

#Phasse III Data Generation

#TimeGane

# Dependency Check

In [ ]:
#cd /home/claude && python3 << 'EOF'
print("\n" + "="*60)
print("🎯 PHASE 3: SYNTHETIC DATA GENERATION")
print("="*60)
print("\nObjective: Generate realistic synthetic sensor data")
print("Method: TimeGAN (Time-series Generative Adversarial Network)")
print("\nPhase 3 Components:")
print("  1. Data Preprocessing & Normalization")
print("  2. TimeGAN Architecture Implementation")
print("  3. Model Training")
print("  4. Synthetic Data Generation")
print("  5. Quality Validation")
print("="*60 + "\n")

# Check required libraries
import subprocess
import sys

print("Checking dependencies...")
required_packages = ['tensorflow', 'scikit-learn', 'numpy', 'pandas', 'matplotlib']

missing = []
for package in required_packages:
    try:
        __import__(package)
        print(f"✅ {package}")
    except ImportError:
        missing.append(package)
        print(f"❌ {package} - MISSING")

if missing:
    print(f"\n⚠️  Installing missing packages: {', '.join(missing)}")
    for package in missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                             package, "--break-system-packages", "--quiet"])
    print("✅ All dependencies installed!")
else:
    print("\n✅ All dependencies available!")



🎯 PHASE 3: SYNTHETIC DATA GENERATION

Objective: Generate realistic synthetic sensor data
Method: TimeGAN (Time-series Generative Adversarial Network)

Phase 3 Components:
  1. Data Preprocessing & Normalization
  2. TimeGAN Architecture Implementation
  3. Model Training
  4. Synthetic Data Generation
  5. Quality Validation

Checking dependencies...
✅ tensorflow
❌ scikit-learn - MISSING
✅ numpy
✅ pandas
✅ matplotlib

⚠️  Installing missing packages: scikit-learn
✅ All dependencies installed!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# we have got the merged csv of cycle activity

#Normalizing the data and creating .npy files for timegan training

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import pickle

print("="*70)
print("PREPARING DATA FOR GAN TRAINING")
print("="*70)

# Step 1: Load merged dataset
print("\n📂 Step 1: Loading merged dataset...")
df = pd.read_csv('/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/merged_sensors_100hz.csv')
print(f"✅ Loaded: {df.shape}")
print(f"   Rows: {df.shape[0]}")
print(f"   Columns: {df.shape[1]}")

# Step 2: Extract features (exclude time column)
print("\n📊 Step 2: Extracting features...")
feature_columns = [col for col in df.columns if col != 'time_seconds']
features = df[feature_columns].values

print(f"✅ Features extracted: {features.shape}")
print(f"   Feature columns ({len(feature_columns)}):")
for i, col in enumerate(feature_columns, 1):
    print(f"      {i:2d}. {col}")

# Step 3: Normalize to [0, 1] range
print("\n🔧 Step 3: Normalizing data to [0, 1] range...")
scaler = MinMaxScaler(feature_range=(0, 1))
features_normalized = scaler.fit_transform(features)

print(f"✅ Normalized:")
print(f"   Original range: [{features.min():.2f}, {features.max():.2f}]")
print(f"   Normalized range: [{features_normalized.min():.4f}, {features_normalized.max():.4f}]")

# Step 4: Create sequences for TimeGAN
print("\n📦 Step 4: Creating sequences for GAN training...")
seq_length = 100  # 1 second at 100 Hz
stride = 50       # 50% overlap

sequences = []
for i in range(0, len(features_normalized) - seq_length, stride):
    seq = features_normalized[i:i+seq_length]
    sequences.append(seq)

sequences = np.array(sequences)
print(f"✅ Created sequences: {sequences.shape}")
print(f"   Number of sequences: {sequences.shape[0]}")
print(f"   Sequence length: {sequences.shape[1]} timesteps")
print(f"   Features per timestep: {sequences.shape[2]}")
print(f"   Total data points: {sequences.size:,}")

# Step 5: Split into train/validation
print("\n✂️  Step 5: Splitting into train/validation sets...")
train_ratio = 0.8
train_size = int(len(sequences) * train_ratio)

train_sequences = sequences[:train_size]
val_sequences = sequences[train_size:]

print(f"✅ Split complete:")
print(f"   Training set: {train_sequences.shape[0]} sequences ({train_ratio*100:.0f}%)")
print(f"   Validation set: {val_sequences.shape[0]} sequences ({(1-train_ratio)*100:.0f}%)")

# Step 6: Save as .npy files
print("\n💾 Step 6: Saving as .npy files...")

# Save sequences
np.save('/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/train_sequences_19features.npy', train_sequences)
np.save('/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/val_sequences_19features.npy', val_sequences)
np.save('/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/all_sequences_19features.npy', sequences)

# Save scaler for denormalization later
with open('/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/scaler_19features.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save original features (unnormalized) for reference
np.save('/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/features_unnormalized.npy', features)

print(f"✅ Saved files:")
print(f"   • train_sequences_19features.npy     : {train_sequences.nbytes / 1024 / 1024:.2f} MB")
print(f"   • val_sequences_19features.npy       : {val_sequences.nbytes / 1024 / 1024:.2f} MB")
print(f"   • all_sequences_19features.npy       : {sequences.nbytes / 1024 / 1024:.2f} MB")
print(f"   • scaler_19features.pkl              : For denormalization")
print(f"   • features_unnormalized.npy          : Original data (reference)")

# Step 7: Print statistics
print("\n📊 Step 7: Data statistics...")
print("\nNormalized Training Data:")
for i, col in enumerate(feature_columns):
    feature_data = train_sequences[:, :, i].flatten()
    print(f"   {col:<20} mean={feature_data.mean():.4f}, std={feature_data.std():.4f}")

# Step 8: Create metadata
print("\n📝 Step 8: Creating metadata file...")
metadata = {
    'total_sequences': int(len(sequences)),
    'train_sequences': int(len(train_sequences)),
    'val_sequences': int(len(val_sequences)),
    'sequence_length': int(seq_length),
    'num_features': int(sequences.shape[2]),
    'sampling_rate_hz': 100,
    'stride': int(stride),
    'normalization': 'MinMaxScaler [0, 1]',
    'feature_columns': feature_columns,
    'data_shape': {
        'sequences': list(sequences.shape),
        'train': list(train_sequences.shape),
        'val': list(val_sequences.shape)
    },
    'file_paths': {
        'train': 'train_sequences_19features.npy',
        'val': 'val_sequences_19features.npy',
        'all': 'all_sequences_19features.npy',
        'scaler': 'scaler_19features.pkl',
        'unnormalized': 'features_unnormalized.npy'
    }
}

import json
with open('/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/gan_training_data_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Saved: gan_training_data_metadata.json")

print("\n" + "="*70)
print("✅ DATA PREPARATION COMPLETE!")
print("="*70)
print(f"\n📦 Ready for GAN Training:")
print(f"   • Training sequences: {train_sequences.shape}")
print(f"   • Validation sequences: {val_sequences.shape}")
print(f"   • Features: {sequences.shape[2]}")
print(f"   • Normalized: [0, 1]")
print(f"   • Format: .npy (NumPy arrays)")
print(f"\n🚀 You can now train TimeGAN with this data!")

PREPARING DATA FOR GAN TRAINING

📂 Step 1: Loading merged dataset...
✅ Loaded: (24037, 20)
   Rows: 24037
   Columns: 20

📊 Step 2: Extracting features...
✅ Features extracted: (24037, 19)
   Feature columns (19):
       1. acc_z
       2. acc_y
       3. acc_x
       4. gyro_z
       5. gyro_y
       6. gyro_x
       7. mag_z
       8. mag_y
       9. mag_x
      10. orient_qz
      11. orient_qy
      12. orient_qx
      13. orient_qw
      14. orient_roll
      15. orient_pitch
      16. orient_yaw
      17. grav_z
      18. grav_y
      19. grav_x

🔧 Step 3: Normalizing data to [0, 1] range...
✅ Normalized:
   Original range: [-95.34, 71.16]
   Normalized range: [0.0000, 1.0000]

📦 Step 4: Creating sequences for GAN training...
✅ Created sequences: (479, 100, 19)
   Number of sequences: 479
   Sequence length: 100 timesteps
   Features per timestep: 19
   Total data points: 910,100

✂️  Step 5: Splitting into train/validation sets...
✅ Split complete:
   Training set: 383 sequences

In [ ]:
!pip install --no-deps timegan

#DataType conversion

In [ ]:
import numpy as np

# Load and check data type
train_data = np.load('/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/train_sequences_19features.npy')
print(f"Current dtype: {train_data.dtype}")

# Convert to float32 (TensorFlow default)
train_data = train_data.astype(np.float32)
val_data = np.load('/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/val_sequences_19features.npy').astype(np.float32)

# Save with correct dtype
np.save('/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/train_sequences_19features.npy', train_data)
np.save('/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/val_sequences_19features.npy', val_data)

print(f"✅ Converted to float32 and saved")
print(f"   Training: {train_data.shape}, dtype={train_data.dtype}")
print(f"   Validation: {val_data.shape}, dtype={val_data.dtype}")


Current dtype: float64
✅ Converted to float32 and saved
   Training: (383, 100, 19), dtype=float32
   Validation: (96, 100, 19), dtype=float32


#Time GAn Implementation first_way

In [ ]:
"""
TimeGAN REBALANCED - Fixes for Discriminator Dominance
Critical changes to prevent mode collapse and variance loss
"""

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import json
import os


class RebalancedTimeGAN:
    """
    Key changes from previous version:
    1. Stronger generator (higher LR, less dropout)
    2. Weaker discriminator (lower LR, more dropout, label noise)
    3. Gradient penalty for discriminator stability
    4. Better loss balancing
    """

    def __init__(self, seq_len, n_features, hidden_dim=96):  # Reduced from 128
        self.seq_len = seq_len
        self.n_features = n_features
        self.hidden_dim = hidden_dim

        self.embedder = self._build_embedder()
        self.recovery = self._build_recovery()
        self.generator = self._build_generator()
        self.discriminator = self._build_discriminator()
        self.supervisor = self._build_supervisor()

        # CRITICAL: Rebalanced learning rates
        self.e_optimizer = keras.optimizers.Adam(0.0005)
        self.r_optimizer = keras.optimizers.Adam(0.0005)
        self.g_optimizer = keras.optimizers.Adam(0.002)   # ← 2.5x higher (was 0.0008)
        self.d_optimizer = keras.optimizers.Adam(0.0001)  # ← 3x lower (was 0.0003)
        self.s_optimizer = keras.optimizers.Adam(0.0005)

    def _build_embedder(self):
        return keras.Sequential([
            layers.Input(shape=(self.seq_len, self.n_features)),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.BatchNormalization(),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.BatchNormalization(),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dense(self.hidden_dim, activation='sigmoid')
        ], name='Embedder')

    def _build_recovery(self):
        return keras.Sequential([
            layers.Input(shape=(self.seq_len, self.hidden_dim)),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.BatchNormalization(),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dense(self.n_features, activation='sigmoid')
        ], name='Recovery')

    def _build_generator(self):
        """Stronger generator - LESS dropout"""
        return keras.Sequential([
            layers.Input(shape=(self.seq_len, self.n_features)),
            layers.GaussianNoise(0.15),  # ← Increased noise for diversity
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dropout(0.2),  # ← Reduced from 0.3
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dropout(0.2),  # ← Reduced from 0.3
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dense(self.hidden_dim, activation='sigmoid')
        ], name='Generator')

    def _build_discriminator(self):
        """Weaker discriminator - MORE dropout"""
        return keras.Sequential([
            layers.Input(shape=(self.seq_len, self.hidden_dim)),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dropout(0.5),  # ← Increased from 0.4
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dropout(0.5),  # ← Increased from 0.4
            layers.Dense(1, activation='sigmoid')
        ], name='Discriminator')

    def _build_supervisor(self):
        return keras.Sequential([
            layers.Input(shape=(self.seq_len - 1, self.hidden_dim)),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dense(self.hidden_dim, activation='sigmoid')
        ], name='Supervisor')

    @tf.function
    def train_embedder(self, X):
        with tf.GradientTape(persistent=True) as tape:
            H = self.embedder(X, training=True)
            X_tilde = self.recovery(H, training=True)
            E_loss_T0 = tf.reduce_mean(tf.abs(X - X_tilde))

            H_mean = tf.reduce_mean(H, axis=0)
            H_var = tf.reduce_mean(tf.square(H - H_mean), axis=0)
            E_loss_0 = tf.reduce_mean(tf.square(H_mean)) + tf.reduce_mean(tf.square(H_var - 1))

            E_loss = 10 * tf.sqrt(E_loss_T0) + 0.1 * E_loss_0

        e_vars = self.embedder.trainable_variables
        r_vars = self.recovery.trainable_variables
        e_grads = tape.gradient(E_loss, e_vars)
        r_grads = tape.gradient(E_loss, r_vars)
        self.e_optimizer.apply_gradients(zip(e_grads, e_vars))
        self.r_optimizer.apply_gradients(zip(r_grads, r_vars))
        del tape
        return E_loss

    @tf.function
    def train_supervisor(self, X):
        with tf.GradientTape() as tape:
            H = self.embedder(X, training=False)
            H_supervise = self.supervisor(H[:, :-1, :], training=True)
            S_loss = tf.reduce_mean(tf.abs(H[:, 1:, :] - H_supervise))

        s_vars = self.supervisor.trainable_variables
        s_grads = tape.gradient(S_loss, s_vars)
        self.s_optimizer.apply_gradients(zip(s_grads, s_vars))
        return S_loss

    @tf.function
    def train_generator_discriminator(self, X, Z):
        batch_size = tf.shape(X)[0]

        # CRITICAL: Label smoothing + noise for discriminator regularization
        real_labels = tf.ones((batch_size, self.seq_len, 1)) * tf.random.uniform((), 0.85, 0.95)
        fake_labels = tf.ones((batch_size, self.seq_len, 1)) * tf.random.uniform((), 0.05, 0.15)
        fake_labels_supervise = tf.ones((batch_size, self.seq_len - 1, 1)) * tf.random.uniform((), 0.05, 0.15)

        with tf.GradientTape(persistent=True) as tape:
            H = self.embedder(X, training=False)
            E_hat = self.generator(Z, training=True)
            H_hat_supervise = self.supervisor(E_hat[:, :-1, :], training=True)

            Y_real = self.discriminator(H, training=True)
            Y_fake = self.discriminator(E_hat, training=True)
            Y_fake_e = self.discriminator(H_hat_supervise, training=True)

            D_loss_real = tf.reduce_mean(tf.keras.losses.binary_crossentropy(real_labels, Y_real))
            D_loss_fake = tf.reduce_mean(tf.keras.losses.binary_crossentropy(fake_labels, Y_fake))
            D_loss_fake_e = tf.reduce_mean(tf.keras.losses.binary_crossentropy(
                fake_labels_supervise, Y_fake_e))
            D_loss = D_loss_real + D_loss_fake + D_loss_fake_e

            # Generator losses with INCREASED weights for variance matching
            G_loss_U = tf.reduce_mean(tf.keras.losses.binary_crossentropy(
                tf.ones((batch_size, self.seq_len, 1)) * 0.9, Y_fake))
            G_loss_U_e = tf.reduce_mean(tf.keras.losses.binary_crossentropy(
                tf.ones((batch_size, self.seq_len - 1, 1)) * 0.9, Y_fake_e))
            G_loss_S = tf.reduce_mean(tf.abs(H_hat_supervise - E_hat[:, 1:, :]))

            # CRITICAL: Stronger moment matching
            E_hat_mean = tf.reduce_mean(E_hat, axis=0)
            H_mean = tf.reduce_mean(H, axis=0)
            E_hat_var = tf.reduce_mean(tf.square(E_hat - E_hat_mean), axis=0)
            H_var = tf.reduce_mean(tf.square(H - H_mean), axis=0)

            G_loss_V1 = tf.reduce_mean(tf.abs(E_hat_mean - H_mean))
            G_loss_V2 = tf.reduce_mean(tf.abs(tf.sqrt(E_hat_var + 1e-6) - tf.sqrt(H_var + 1e-6)))

            # CRITICAL: Increased variance matching weight (200 vs 100)
            G_loss = G_loss_U + G_loss_U_e + 100 * tf.sqrt(G_loss_S) + 200 * G_loss_V1 + 200 * G_loss_V2

        # Update discriminator
        d_vars = self.discriminator.trainable_variables
        d_grads = tape.gradient(D_loss, d_vars)
        # Gradient clipping for discriminator stability
        d_grads = [tf.clip_by_norm(g, 1.0) for g in d_grads]
        self.d_optimizer.apply_gradients(zip(d_grads, d_vars))

        # Update generator and supervisor
        g_vars = self.generator.trainable_variables + self.supervisor.trainable_variables
        g_grads = tape.gradient(G_loss, g_vars)
        self.g_optimizer.apply_gradients(zip(g_grads[:len(self.generator.trainable_variables)],
                                              self.generator.trainable_variables))
        self.s_optimizer.apply_gradients(zip(g_grads[len(self.generator.trainable_variables):],
                                              self.supervisor.trainable_variables))
        del tape
        return D_loss, G_loss

    def generate(self, n_samples):
        Z = tf.random.normal((n_samples, self.seq_len, self.n_features))
        E_hat = self.generator(Z, training=False)
        X_hat = self.recovery(E_hat, training=False)
        return X_hat.numpy()

    def save_models(self, path):
        os.makedirs(path, exist_ok=True)
        self.embedder.save_weights(f'{path}/embedder.weights.h5')
        self.recovery.save_weights(f'{path}/recovery.weights.h5')
        self.generator.save_weights(f'{path}/generator.weights.h5')
        self.discriminator.save_weights(f'{path}/discriminator.weights.h5')
        self.supervisor.save_weights(f'{path}/supervisor.weights.h5')


def train_rebalanced_timegan(train_data, val_data,
                             hidden_dim=96,           # Reduced from 128
                             embedder_epochs=75,
                             supervisor_epochs=40,
                             gan_epochs=200,          # Increased from 150
                             batch_size=16,
                             save_path='timegan_rebalanced'):

    seq_len, n_features = train_data.shape[1], train_data.shape[2]

    print("="*70)
    print("REBALANCED TIMEGAN - FIX FOR DISCRIMINATOR DOMINANCE")
    print("="*70)
    print(f"\nConfiguration:")
    print(f"  Hidden dim: {hidden_dim} (reduced from 128)")
    print(f"  GAN epochs: {gan_epochs} (increased from 150)")
    print(f"  G learning rate: 0.002 (was 0.0008)")
    print(f"  D learning rate: 0.0001 (was 0.0003)")
    print(f"\nKey Changes:")
    print(f"  ✓ Generator 2.5x stronger (higher LR, less dropout)")
    print(f"  ✓ Discriminator 3x weaker (lower LR, more dropout)")
    print(f"  ✓ Label noise for D regularization")
    print(f"  ✓ 2x stronger variance matching (200 vs 100)")
    print(f"  ✓ Gradient clipping for stability")

    model = RebalancedTimeGAN(seq_len, n_features, hidden_dim)

    history = {
        'embedder_loss': [],
        'supervisor_loss': [],
        'discriminator_loss': [],
        'generator_loss': []
    }

    n_batches = len(train_data) // batch_size

    # Phase 1
    print("\n" + "="*70)
    print("PHASE 1: EMBEDDER & RECOVERY")
    print("="*70)
    for epoch in range(embedder_epochs):
        epoch_loss = 0
        for _ in range(n_batches):
            idx = np.random.choice(len(train_data), batch_size, replace=False)
            loss = model.train_embedder(train_data[idx])
            epoch_loss += loss

        avg_loss = epoch_loss / n_batches
        history['embedder_loss'].append(float(avg_loss))

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}/{embedder_epochs}: E_loss={avg_loss:.4f}")

    # Phase 2
    print("\n" + "="*70)
    print("PHASE 2: SUPERVISOR")
    print("="*70)
    for epoch in range(supervisor_epochs):
        epoch_loss = 0
        for _ in range(n_batches):
            idx = np.random.choice(len(train_data), batch_size, replace=False)
            loss = model.train_supervisor(train_data[idx])
            epoch_loss += loss

        avg_loss = epoch_loss / n_batches
        history['supervisor_loss'].append(float(avg_loss))

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}/{supervisor_epochs}: S_loss={avg_loss:.4f}")

    # Phase 3
    print("\n" + "="*70)
    print("PHASE 3: GAN TRAINING (REBALANCED)")
    print("="*70)
    for epoch in range(gan_epochs):
        epoch_d_loss, epoch_g_loss = 0, 0

        for _ in range(n_batches):
            idx = np.random.choice(len(train_data), batch_size, replace=False)
            X_batch = train_data[idx]
            Z_batch = np.random.normal(size=(batch_size, seq_len, n_features))

            # Train generator MORE (3x per discriminator)
            for _ in range(3):  # Increased from 2
                _, g_loss = model.train_generator_discriminator(X_batch, Z_batch)
                epoch_g_loss += g_loss

            d_loss, _ = model.train_generator_discriminator(X_batch, Z_batch)
            epoch_d_loss += d_loss

        avg_d = epoch_d_loss / n_batches
        avg_g = epoch_g_loss / (n_batches * 3)
        history['discriminator_loss'].append(float(avg_d))
        history['generator_loss'].append(float(avg_g))

        if (epoch + 1) % 10 == 0:
            ratio = avg_g / (avg_d + 1e-10)
            print(f"  Epoch {epoch+1:3d}/{gan_epochs}: D_loss={avg_d:.4f}, G_loss={avg_g:.4f}, G/D={ratio:.2f}")

            # Quality check
            if epoch > 50:
                if avg_d < 0.3:
                    print(f"    ⚠️  D too strong ({avg_d:.4f}), consider early stop")
                if avg_g > 20:
                    print(f"    ⚠️  G struggling ({avg_g:.4f}), may need adjustment")

    print("\n💾 Saving model...")
    model.save_models(save_path)

    with open(f'{save_path}/training_history.json', 'w') as f:
        json.dump(history, f, indent=2)

    print("\n" + "="*70)
    print("✅ TRAINING COMPLETE!")
    print("="*70)
    print(f"\nFinal losses:")
    print(f"  Embedder: {history['embedder_loss'][-1]:.4f}")
    print(f"  Supervisor: {history['supervisor_loss'][-1]:.4f}")
    print(f"  Discriminator: {history['discriminator_loss'][-1]:.4f}")
    print(f"  Generator: {history['generator_loss'][-1]:.4f}")
    print(f"  G/D Ratio: {history['generator_loss'][-1] / history['discriminator_loss'][-1]:.2f}")

    final_d = history['discriminator_loss'][-1]
    final_g = history['generator_loss'][-1]
    ratio = final_g / final_d

    print(f"\n📊 Quality Check:")
    if 0.5 < final_d < 1.2 and 3 < final_g < 8 and 3 < ratio < 10:
        print(f"  ✅ Losses look good! Expect better synthetic data.")
    else:
        print(f"  ⚠️  Losses still concerning. Run validation to confirm.")

    return model, history


if __name__ == "__main__":

    BASE_PATH = '/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling'
    DATA_PATH = f'{BASE_PATH}/train_sequences_19features.npy'
    VAL_PATH = f'{BASE_PATH}/val_sequences_19features.npy'
    SAVE_PATH = f'{BASE_PATH}/timegan_rebalanced_v2'

    print("Loading data...")
    train_data = np.load(DATA_PATH)
    val_data = np.load(VAL_PATH)
    print(f"Loaded: train={train_data.shape}, val={val_data.shape}\n")

    model, history = train_rebalanced_timegan(
        train_data=train_data,
        val_data=val_data,
        hidden_dim=96,
        embedder_epochs=75,
        supervisor_epochs=40,
        gan_epochs=200,
        batch_size=16,
        save_path=SAVE_PATH
    )

    print("\nTesting generation...")
    test_samples = model.generate(20)
    np.save(f'{SAVE_PATH}/test_samples.npy', test_samples)
    print(f"Generated: {test_samples.shape}")
    print(f"Range: [{test_samples.min():.4f}, {test_samples.max():.4f}]")

    print("\n✅ Ready for validation!")

Loading data...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/train_sequences_19features.npy'

#Quck validation along with synthetic data generation

In [ ]:
"""
Quick Validation: Check if synthetic data is usable despite high G_loss
FIXED: Proper model loading without import issues
"""

import numpy as np
import pickle
import matplotlib.pyplot as plt
from scipy import stats
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Paths - ADJUST THESE
BASE_PATH = '/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling'
REAL_NORM_PATH = f'{BASE_PATH}/train_sequences_19features.npy'
SCALER_PATH = f'{BASE_PATH}/scaler_19features.pkl'
MODEL_PATH = f'{BASE_PATH}/timegan_rebalanced_v2'

print("="*70)
print("QUICK VALIDATION CHECK")
print("="*70)

# Define the model class inline (no import needed)
class ImprovedTimeGAN:

    def __init__(self, seq_len, n_features, hidden_dim=128):
        self.seq_len = seq_len
        self.n_features = n_features
        self.hidden_dim = hidden_dim

        self.embedder = self._build_embedder()
        self.recovery = self._build_recovery()
        self.generator = self._build_generator()
        self.discriminator = self._build_discriminator()
        self.supervisor = self._build_supervisor()

    def _build_embedder(self):
        return keras.Sequential([
            layers.Input(shape=(self.seq_len, self.n_features)),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.BatchNormalization(),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.BatchNormalization(),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dense(self.hidden_dim, activation='sigmoid')
        ], name='Embedder')

    def _build_recovery(self):
        return keras.Sequential([
            layers.Input(shape=(self.seq_len, self.hidden_dim)),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.BatchNormalization(),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dense(self.n_features, activation='sigmoid')
        ], name='Recovery')

    def _build_generator(self):
        return keras.Sequential([
            layers.Input(shape=(self.seq_len, self.n_features)),
            layers.GaussianNoise(0.1),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dropout(0.3),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dropout(0.3),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dense(self.hidden_dim, activation='sigmoid')
        ], name='Generator')

    def _build_discriminator(self):
        return keras.Sequential([
            layers.Input(shape=(self.seq_len, self.hidden_dim)),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dropout(0.4),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dropout(0.4),
            layers.Dense(1, activation='sigmoid')
        ], name='Discriminator')

    def _build_supervisor(self):
        return keras.Sequential([
            layers.Input(shape=(self.seq_len - 1, self.hidden_dim)),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dense(self.hidden_dim, activation='sigmoid')
        ], name='Supervisor')

    def generate(self, n_samples):
        """Generate synthetic sequences"""
        Z = tf.random.normal((n_samples, self.seq_len, self.n_features))
        E_hat = self.generator(Z, training=False)
        X_hat = self.recovery(E_hat, training=False)
        return X_hat.numpy()

    def load_models(self, path):
        """Load all model weights"""
        self.embedder.load_weights(f'{path}/embedder.weights.h5')
        self.recovery.load_weights(f'{path}/recovery.weights.h5')
        self.generator.load_weights(f'{path}/generator.weights.h5')
        self.discriminator.load_weights(f'{path}/discriminator.weights.h5')
        self.supervisor.load_weights(f'{path}/supervisor.weights.h5')


# 1. Load and generate
print("\n1. Loading model and generating synthetic data...")
real_norm = np.load(REAL_NORM_PATH)
print(f"   Real data: {real_norm.shape}")

# Initialize and load model
model = ImprovedTimeGAN(seq_len=100, n_features=19, hidden_dim=128)
print("   Loading model weights...")
model.load_models(MODEL_PATH)
print("   ✓ Model loaded")

# Generate synthetic (normalized)
print("   Generating synthetic sequences...")
n_synthetic = len(real_norm)
synthetic_norm = model.generate(n_synthetic)
print(f"   Generated: {synthetic_norm.shape}")

# 2. Denormalize both
print("\n2. Denormalizing data...")
with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)

real_denorm = scaler.inverse_transform(real_norm.reshape(-1, 19)).reshape(real_norm.shape)
synthetic_denorm = scaler.inverse_transform(synthetic_norm.reshape(-1, 19)).reshape(synthetic_norm.shape)

print(f"   Real range: [{real_denorm.min():.2f}, {real_denorm.max():.2f}]")
print(f"   Synth range: [{synthetic_denorm.min():.2f}, {synthetic_denorm.max():.2f}]")

# 3. Quick statistics check (motion sensors only)
print("\n3. Motion Sensors Statistics (most important):")
print("="*70)

feature_names = ['acc_z', 'acc_y', 'acc_x', 'gyro_z', 'gyro_y', 'gyro_x']
passed = 0
total = 6

for i, name in enumerate(feature_names):
    real_mean = real_denorm[:, :, i].mean()
    synth_mean = synthetic_denorm[:, :, i].mean()

    real_std = real_denorm[:, :, i].std()
    synth_std = synthetic_denorm[:, :, i].std()

    mean_diff = abs(real_mean - synth_mean) / (abs(real_mean) + 1e-10) * 100
    std_diff = abs(real_std - synth_std) / (abs(real_std) + 1e-10) * 100

    status = "✅" if mean_diff < 50 and std_diff < 50 else "❌"
    if mean_diff < 50 and std_diff < 50:
        passed += 1

    print(f"{name:<10} Real: μ={real_mean:>7.2f} σ={real_std:>6.2f} | "
          f"Synth: μ={synth_mean:>7.2f} σ={synth_std:>6.2f} | "
          f"Δμ={mean_diff:>5.1f}% Δσ={std_diff:>5.1f}% {status}")

print(f"\n📊 Motion sensors passing: {passed}/{total}")

# 4. Distribution similarity (KS test)
print("\n4. Distribution Similarity (KS Test):")
print("="*70)

ks_passed = 0
for i, name in enumerate(feature_names):
    real_flat = real_denorm[:, :, i].flatten()
    synth_flat = synthetic_denorm[:, :, i].flatten()

    ks_stat, p_value = stats.ks_2samp(real_flat, synth_flat)
    status = "✅" if p_value > 0.01 else "❌"  # Relaxed threshold (0.01 instead of 0.05)
    if p_value > 0.01:
        ks_passed += 1

    print(f"{name:<10} KS={ks_stat:.4f}, p={p_value:.4f} {status}")

print(f"\n📊 KS test passing: {ks_passed}/{total}")

# 5. Visual check (first sequence)
print("\n5. Generating visual comparison...")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Real vs Synthetic - First Sequence Comparison', fontsize=16, fontweight='bold')

for idx, (ax, name) in enumerate(zip(axes.flat, feature_names)):
    ax.plot(real_denorm[0, :, idx], label='Real', linewidth=2, alpha=0.8)
    ax.plot(synthetic_denorm[0, :, idx], label='Synthetic', linewidth=2, alpha=0.8, linestyle='--')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Time')
    ax.set_ylabel('Value')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
output_img = f'{MODEL_PATH}/quick_validation.png'
plt.savefig(output_img, dpi=150, bbox_inches='tight')
print(f"   ✓ Saved: quick_validation.png")
plt.close()

# 6. Overall verdict
print("\n" + "="*70)
print("OVERALL VERDICT")
print("="*70)

total_score = passed + ks_passed
max_score = total * 2

print(f"\nScore: {total_score}/{max_score} ({total_score/max_score*100:.0f}%)")

if total_score >= 9:  # 75%+
    print("\n✅ PASS: Synthetic data is USABLE")
    print("   Quality is acceptable despite high G_loss")
    print("   Proceed with anti-emulation detection")
    verdict = "PASS"
elif total_score >= 6:  # 50-75%
    print("\n⚠️  MARGINAL: Synthetic data has issues but may work")
    print("   Consider retraining with adjustments")
    print("   Can proceed cautiously")
    verdict = "MARGINAL"
else:  # <50%
    print("\n❌ FAIL: Synthetic data quality too poor")
    print("   MUST retrain with different hyperparameters")
    print("   Do NOT proceed to detection")
    verdict = "FAIL"

# 7. Save results
print(f"\n💾 Saving synthetic data...")
np.save(f'{MODEL_PATH}/synthetic_normalized.npy', synthetic_norm)
np.save(f'{MODEL_PATH}/synthetic_denormalized.npy', synthetic_denorm)
print(f"   ✓ Saved to: {MODEL_PATH}/")

print("\n" + "="*70)
print("RECOMMENDATIONS")
print("="*70)

if verdict == "PASS":
    print("\n✅ Next steps:")
    print("   1. Run full validation (7 tests)")
    print("   2. Build anti-emulation classifier")
    print("   3. Evaluate on test set")

elif verdict == "MARGINAL":
    print("\n⚠️  Options:")
    print("   Option A: Use current data (faster, may work)")
    print("   Option B: Retrain with adjusted hyperparameters:")
    print("      - Increase G learning rate: 0.0008 → 0.0015")
    print("      - Reduce D learning rate: 0.0003 → 0.0001")
    print("      - More GAN epochs: 150 → 200")

else:  # FAIL
    print("\n❌ Required changes:")
    print("   1. Adjust hyperparameters:")
    print("      - G learning rate: 0.0008 → 0.002")
    print("      - D learning rate: 0.0003 → 0.0001")
    print("      - Reduce discriminator dropout: 0.4 → 0.2")
    print("      - GAN epochs: 150 → 250")
    print("   2. Consider reducing hidden_dim: 128 → 96")
    print("   3. Try gradient penalty for discriminator")

print("\n✅ Validation complete!")
print(f"\nFiles saved:")
print(f"  - {MODEL_PATH}/synthetic_normalized.npy")
print(f"  - {MODEL_PATH}/synthetic_denormalized.npy")
print(f"  - {MODEL_PATH}/quick_validation.png")

QUICK VALIDATION CHECK

1. Loading model and generating synthetic data...
   Real data: (383, 100, 19)
   Loading model weights...
   ✓ Model loaded
   Generating synthetic sequences...
   Generated: (383, 100, 19)

2. Denormalizing data...
   Real range: [-95.34, 58.69]
   Synth range: [-26.40, 29.29]

3. Motion Sensors Statistics (most important):
acc_z      Real: μ=   0.08 σ=  1.02 | Synth: μ=  -0.73 σ=  0.23 | Δμ=970.3% Δσ= 77.5% ❌
acc_y      Real: μ=  -0.04 σ=  1.80 | Synth: μ=   0.68 σ=  0.30 | Δμ=1646.1% Δσ= 83.1% ❌
acc_x      Real: μ=  -0.02 σ=  1.41 | Synth: μ=   0.55 σ=  0.32 | Δμ=2578.0% Δσ= 77.4% ❌
gyro_z     Real: μ=   0.01 σ=  0.50 | Synth: μ=   0.15 σ=  0.08 | Δμ=2646.4% Δσ= 83.8% ❌
gyro_y     Real: μ=  -0.02 σ=  0.59 | Synth: μ=  -0.03 σ=  0.06 | Δμ=109.3% Δσ= 89.1% ❌
gyro_x     Real: μ=  -0.00 σ=  0.33 | Synth: μ=   0.09 σ=  0.01 | Δμ=2545.7% Δσ= 96.1% ❌

📊 Motion sensors passing: 0/6

4. Distribution Similarity (KS Test):
acc_z      KS=0.7998, p=0.0000 ❌
acc_y      KS

# Generate the synthetic data script

In [ ]:
"""
Synthetic Data Generation Script
Load trained TimeGAN model and generate synthetic sensor data
"""

import numpy as np
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import pandas as pd
import os


class TimeGAN:
    """TimeGAN model - same architecture as training"""

    def __init__(self, seq_len, n_features, hidden_dim=64):
        self.seq_len = seq_len
        self.n_features = n_features
        self.hidden_dim = hidden_dim

        self.embedder = self._build_embedder()
        self.recovery = self._build_recovery()
        self.generator = self._build_generator()
        self.discriminator = self._build_discriminator()
        self.supervisor = self._build_supervisor()

    def _build_embedder(self):
        return keras.Sequential([
            layers.Input(shape=(self.seq_len, self.n_features)),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dense(self.hidden_dim, activation='sigmoid')
        ], name='Embedder')

    def _build_recovery(self):
        return keras.Sequential([
            layers.Input(shape=(self.seq_len, self.hidden_dim)),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dense(self.n_features, activation='sigmoid')
        ], name='Recovery')

    def _build_generator(self):
        return keras.Sequential([
            layers.Input(shape=(self.seq_len, self.n_features)),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dropout(0.2),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dropout(0.2),
            layers.Dense(self.hidden_dim, activation='sigmoid')
        ], name='Generator')

    def _build_discriminator(self):
        return keras.Sequential([
            layers.Input(shape=(self.seq_len, self.hidden_dim)),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dropout(0.3),
            layers.Dense(1, activation='sigmoid')
        ], name='Discriminator')

    def _build_supervisor(self):
        return keras.Sequential([
            layers.Input(shape=(self.seq_len - 1, self.hidden_dim)),
            layers.LSTM(self.hidden_dim, return_sequences=True),
            layers.Dense(self.hidden_dim, activation='sigmoid')
        ], name='Supervisor')

    def load_models(self, path):
        """Load trained model weights"""
        self.embedder.load_weights(f'{path}/embedder.weights.h5')
        self.recovery.load_weights(f'{path}/recovery.weights.h5')
        self.generator.load_weights(f'{path}/generator.weights.h5')
        self.discriminator.load_weights(f'{path}/discriminator.weights.h5')
        self.supervisor.load_weights(f'{path}/supervisor.weights.h5')
        print(f"✅ Loaded model weights from {path}/")

    def generate(self, n_samples):
        """Generate synthetic sequences"""
        Z = tf.random.normal((n_samples, self.seq_len, self.n_features))
        E_hat = self.generator(Z, training=False)
        X_hat = self.recovery(E_hat, training=False)
        return X_hat.numpy()


def generate_synthetic_data(model_path, scaler_path, n_samples=500, output_path='synthetic_output'):
    """
    Complete pipeline: Load model → Generate → Denormalize → Save

    Args:
        model_path: Path to trained model directory
        scaler_path: Path to scaler pickle file
        n_samples: Number of sequences to generate
        output_path: Directory to save outputs
    """

    print("="*70)
    print("SYNTHETIC DATA GENERATION")
    print("="*70)

    # Create output directory
    os.makedirs(output_path, exist_ok=True)

    # Step 1: Load model
    print("\n📦 Step 1: Loading trained TimeGAN model...")
    model = TimeGAN(seq_len=100, n_features=19, hidden_dim=64)
    model.load_models(model_path)

    # Step 2: Generate normalized data
    print(f"\n🎨 Step 2: Generating {n_samples} synthetic sequences...")
    synthetic_normalized = model.generate(n_samples)
    print(f"✅ Generated: {synthetic_normalized.shape}")
    print(f"   Normalized range: [{synthetic_normalized.min():.4f}, {synthetic_normalized.max():.4f}]")

    # Step 3: Load scaler and denormalize
    print("\n🔧 Step 3: Denormalizing to original scale...")
    with open(scaler_path, 'rb') as f:
        scaler = pickle.load(f)

    original_shape = synthetic_normalized.shape
    reshaped = synthetic_normalized.reshape(-1, original_shape[-1])
    synthetic_denorm = scaler.inverse_transform(reshaped)
    synthetic_denorm = synthetic_denorm.reshape(original_shape)

    print(f"✅ Denormalized: {synthetic_denorm.shape}")
    print(f"   Original scale range: [{synthetic_denorm.min():.2f}, {synthetic_denorm.max():.2f}]")

    # Step 4: Save
    print(f"\n💾 Step 4: Saving outputs to {output_path}/...")
    np.save(f'{output_path}/synthetic_normalized_{n_samples}.npy', synthetic_normalized)
    np.save(f'{output_path}/synthetic_denormalized_{n_samples}.npy', synthetic_denorm)
    print(f"✅ Saved .npy files")

    # Step 5: Statistics
    print("\n📊 Step 5: Computing statistics...")
    feature_names = ['acc_z', 'acc_y', 'acc_x', 'gyro_z', 'gyro_y', 'gyro_x',
                     'mag_z', 'mag_y', 'mag_x', 'orient_qz', 'orient_qy', 'orient_qx',
                     'orient_qw', 'orient_roll', 'orient_pitch', 'orient_yaw',
                     'grav_z', 'grav_y', 'grav_x']

    stats = []
    for i, name in enumerate(feature_names):
        feature_data = synthetic_denorm[:, :, i].flatten()
        stats.append({
            'feature': name,
            'mean': float(feature_data.mean()),
            'std': float(feature_data.std()),
            'min': float(feature_data.min()),
            'max': float(feature_data.max())
        })

    stats_df = pd.DataFrame(stats)
    stats_df.to_csv(f'{output_path}/synthetic_statistics.csv', index=False)
    print(f"✅ Saved statistics to synthetic_statistics.csv")

    print("\n" + "="*70)
    print("✅ GENERATION COMPLETE!")
    print("="*70)
    print(f"\nOutputs in {output_path}/:")
    print(f"  - synthetic_normalized_{n_samples}.npy ({n_samples} sequences)")
    print(f"  - synthetic_denormalized_{n_samples}.npy ({n_samples} sequences)")
    print(f"  - synthetic_statistics.csv")

    return synthetic_normalized, synthetic_denorm


def visualize_samples(synthetic_denorm, n_samples=5, output_path='synthetic_output'):
    """Create visualizations of generated samples"""

    print("\n📊 Creating visualizations...")

    feature_names = ['acc_z', 'acc_y', 'acc_x', 'gyro_z', 'gyro_y', 'gyro_x',
                     'mag_z', 'mag_y', 'mag_x', 'orient_qz', 'orient_qy', 'orient_qx',
                     'orient_qw', 'orient_roll', 'orient_pitch', 'orient_yaw',
                     'grav_z', 'grav_y', 'grav_x']

    # Plot 1: First 5 sequences - Accelerometer
    fig, axes = plt.subplots(n_samples, 1, figsize=(14, 10), sharex=True)
    for i in range(n_samples):
        axes[i].plot(synthetic_denorm[i, :, 0], label='acc_z', alpha=0.7)
        axes[i].plot(synthetic_denorm[i, :, 1], label='acc_y', alpha=0.7)
        axes[i].plot(synthetic_denorm[i, :, 2], label='acc_x', alpha=0.7)
        axes[i].set_ylabel(f'Seq {i+1}', fontweight='bold')
        axes[i].grid(True, alpha=0.3)
        if i == 0:
            axes[i].legend(loc='upper right', ncol=3)
    axes[0].set_title('Generated Accelerometer Data (First 5 Sequences)', fontsize=14, fontweight='bold')
    axes[-1].set_xlabel('Timesteps', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_path}/generated_accelerometer.png', dpi=150, bbox_inches='tight')
    plt.close()

    # Plot 2: First 5 sequences - Gyroscope
    fig, axes = plt.subplots(n_samples, 1, figsize=(14, 10), sharex=True)
    for i in range(n_samples):
        axes[i].plot(synthetic_denorm[i, :, 3], label='gyro_z', alpha=0.7)
        axes[i].plot(synthetic_denorm[i, :, 4], label='gyro_y', alpha=0.7)
        axes[i].plot(synthetic_denorm[i, :, 5], label='gyro_x', alpha=0.7)
        axes[i].set_ylabel(f'Seq {i+1}', fontweight='bold')
        axes[i].grid(True, alpha=0.3)
        if i == 0:
            axes[i].legend(loc='upper right', ncol=3)
    axes[0].set_title('Generated Gyroscope Data (First 5 Sequences)', fontsize=14, fontweight='bold')
    axes[-1].set_xlabel('Timesteps', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_path}/generated_gyroscope.png', dpi=150, bbox_inches='tight')
    plt.close()

    # Plot 3: Single sequence - All sensors
    fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)

    sensor_groups = [
        ('Accelerometer (m/s²)', [0, 1, 2], ['acc_z', 'acc_y', 'acc_x']),
        ('Gyroscope (rad/s)', [3, 4, 5], ['gyro_z', 'gyro_y', 'gyro_x']),
        ('Magnetometer (μT)', [6, 7, 8], ['mag_z', 'mag_y', 'mag_x']),
        ('Orientation (degrees)', [13, 14, 15], ['roll', 'pitch', 'yaw']),
        ('Gravity (m/s²)', [16, 17, 18], ['grav_z', 'grav_y', 'grav_x'])
    ]

    for ax, (title, indices, labels) in zip(axes, sensor_groups):
        for idx, label in zip(indices, labels):
            ax.plot(synthetic_denorm[0, :, idx], label=label, alpha=0.8, linewidth=1.2)
        ax.set_ylabel(title, fontweight='bold')
        ax.legend(loc='upper right', ncol=3)
        ax.grid(True, alpha=0.3)

    axes[0].set_title('Generated Multi-Sensor Data (Single Sequence)', fontsize=14, fontweight='bold')
    axes[-1].set_xlabel('Timesteps (100 Hz sampling)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_path}/generated_all_sensors.png', dpi=150, bbox_inches='tight')
    plt.close()

    print(f"✅ Saved 3 visualization plots to {output_path}/")


def export_to_csv(synthetic_denorm, output_path='synthetic_output', max_sequences=10):
    """Export first N sequences to CSV for inspection"""

    print(f"\n📄 Exporting first {max_sequences} sequences to CSV...")

    feature_names = ['acc_z', 'acc_y', 'acc_x', 'gyro_z', 'gyro_y', 'gyro_x',
                     'mag_z', 'mag_y', 'mag_x', 'orient_qz', 'orient_qy', 'orient_qx',
                     'orient_qw', 'orient_roll', 'orient_pitch', 'orient_yaw',
                     'grav_z', 'grav_y', 'grav_x']

    for seq_idx in range(min(max_sequences, len(synthetic_denorm))):
        df = pd.DataFrame(synthetic_denorm[seq_idx], columns=feature_names)
        df.insert(0, 'timestep', range(len(df)))
        df.to_csv(f'{output_path}/sequence_{seq_idx+1}.csv', index=False)

    print(f"✅ Exported {min(max_sequences, len(synthetic_denorm))} CSV files")


def compare_with_real(real_data_path, synthetic_denorm, output_path='synthetic_output'):
    """Compare real vs synthetic statistics"""

    print("\n📊 Comparing with real data...")

    real_data = np.load(real_data_path)

    feature_names = ['acc_z', 'acc_y', 'acc_x', 'gyro_z', 'gyro_y', 'gyro_x',
                     'mag_z', 'mag_y', 'mag_x', 'orient_qz', 'orient_qy', 'orient_qx',
                     'orient_qw', 'orient_roll', 'orient_pitch', 'orient_yaw',
                     'grav_z', 'grav_y', 'grav_x']

    # Load scaler to denormalize real data
    comparison = []
    for i, name in enumerate(feature_names):
        # Real data is normalized, need to denormalize first
        real_feature = real_data[:, :, i].flatten()
        synth_feature = synthetic_denorm[:, :, i].flatten()

        comparison.append({
            'feature': name,
            'real_mean': 'N/A (normalized)',
            'synth_mean': float(synth_feature.mean()),
            'real_std': 'N/A (normalized)',
            'synth_std': float(synth_feature.std()),
            'mean_diff': 'N/A',
            'std_diff': 'N/A'
        })

    comp_df = pd.DataFrame(comparison)
    comp_df.to_csv(f'{output_path}/real_vs_synthetic_comparison.csv', index=False)
    print(f"✅ Saved comparison to real_vs_synthetic_comparison.csv")

     # Path to your trained model directory
    MODEL_PATH = '/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/timegan_19features_improved'

    # Path to your scaler file
    SCALER_PATH = '/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/scaler_19features.pkl'

    # Path to real training data (optional, for comparison)
    REAL_DATA_PATH = '/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/train_sequences_19features.npy'

    # Output directory for synthetic data
    OUTPUT_PATH = '/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/synthetic_output'

    # Number of sequences to generate
    N_SAMPLES = 383

    # ========== END CONFIGURATION ==========


    # Generate synthetic data
    synthetic_norm, synthetic_denorm = generate_synthetic_data(
        model_path=MODEL_PATH,
        scaler_path=SCALER_PATH,
        n_samples=N_SAMPLES,
        output_path=OUTPUT_PATH
    )

    # Create visualizations
    visualize_samples(synthetic_denorm, n_samples=5, output_path=OUTPUT_PATH)

    # Export samples to CSV
    export_to_csv(synthetic_denorm, output_path=OUTPUT_PATH, max_sequences=10)

    # Optional: Compare with real data
    try:
        compare_with_real(REAL_DATA_PATH, synthetic_denorm, output_path=OUTPUT_PATH)
    except:
        print("\n⚠️  Could not load real data for comparison (optional step)")

    print("\n" + "="*70)
    print("🎉 ALL DONE!")
    print("="*70)
    print(f"\nGenerated {N_SAMPLES} synthetic sequences")
    print(f"Check {OUTPUT_PATH}/ for all outputs")

#creating validation script for real v/s syntheitc data generation

In [ ]:
"""
Comprehensive Validation: Real vs Synthetic Data
Multiple statistical tests and visualizations to validate synthetic data quality
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.fft import fft, fftfreq
from scipy.spatial.distance import jensenshannon
import pickle
import os


def load_data(real_path, synthetic_path, scaler_path):
    """Load and prepare data"""
    print("="*70)
    print("LOADING DATA")
    print("="*70)

    # Load real data (normalized)
    real_norm = np.load(real_path)
    print(f"✅ Real data (normalized): {real_norm.shape}")

    # Load synthetic data (denormalized)
    synthetic_denorm = np.load(synthetic_path)
    print(f"✅ Synthetic data (denormalized): {synthetic_denorm.shape}")

    # Load scaler
    with open(scaler_path, 'rb') as f:
        scaler = pickle.load(f)

    # Denormalize real data for comparison
    real_shape = real_norm.shape
    real_reshaped = real_norm.reshape(-1, real_shape[-1])
    real_denorm = scaler.inverse_transform(real_reshaped)
    real_denorm = real_denorm.reshape(real_shape)
    print(f"✅ Real data (denormalized): {real_denorm.shape}")

    return real_norm, real_denorm, synthetic_denorm


def test_1_basic_statistics(real, synthetic, output_path):
    """Test 1: Compare basic statistics (mean, std, min, max)"""

    print("\n" + "="*70)
    print("TEST 1: BASIC STATISTICS COMPARISON")
    print("="*70)

    feature_names = ['acc_z', 'acc_y', 'acc_x', 'gyro_z', 'gyro_y', 'gyro_x',
                     'mag_z', 'mag_y', 'mag_x', 'orient_qz', 'orient_qy', 'orient_qx',
                     'orient_qw', 'orient_roll', 'orient_pitch', 'orient_yaw',
                     'grav_z', 'grav_y', 'grav_x']

    results = []
    for i, name in enumerate(feature_names):
        real_feat = real[:, :, i].flatten()
        synth_feat = synthetic[:, :, i].flatten()

        real_mean, real_std = real_feat.mean(), real_feat.std()
        synth_mean, synth_std = synth_feat.mean(), synth_feat.std()

        mean_diff_pct = abs(real_mean - synth_mean) / (abs(real_mean) + 1e-10) * 100
        std_diff_pct = abs(real_std - synth_std) / (real_std + 1e-10) * 100

        results.append({
            'feature': name,
            'real_mean': real_mean,
            'synth_mean': synth_mean,
            'mean_diff_%': mean_diff_pct,
            'real_std': real_std,
            'synth_std': synth_std,
            'std_diff_%': std_diff_pct,
            'real_min': real_feat.min(),
            'synth_min': synth_feat.min(),
            'real_max': real_feat.max(),
            'synth_max': synth_feat.max()
        })

        status = "✅ GOOD" if mean_diff_pct < 20 and std_diff_pct < 30 else "⚠️ CHECK"
        print(f"{name:<20} Mean: {real_mean:>8.2f} vs {synth_mean:>8.2f} ({mean_diff_pct:>5.1f}%)  {status}")

    df = pd.DataFrame(results)
    df.to_csv(f'{output_path}/test1_basic_statistics.csv', index=False)

    # Summary
    good_count = sum(1 for r in results if r['mean_diff_%'] < 20 and r['std_diff_%'] < 30)
    print(f"\n📊 Summary: {good_count}/{len(results)} features within acceptable range")
    print(f"✅ Saved: test1_basic_statistics.csv")

    return df


def test_2_distribution_similarity(real, synthetic, output_path):
    """Test 2: KS test and Jensen-Shannon divergence for distribution similarity"""

    print("\n" + "="*70)
    print("TEST 2: DISTRIBUTION SIMILARITY")
    print("="*70)

    feature_names = ['acc_z', 'acc_y', 'acc_x', 'gyro_z', 'gyro_y', 'gyro_x',
                     'mag_z', 'mag_y', 'mag_x', 'orient_qz', 'orient_qy', 'orient_qx',
                     'orient_qw', 'orient_roll', 'orient_pitch', 'orient_yaw',
                     'grav_z', 'grav_y', 'grav_x']

    results = []
    for i, name in enumerate(feature_names):
        real_feat = real[:, :, i].flatten()
        synth_feat = synthetic[:, :, i].flatten()

        # Kolmogorov-Smirnov test
        ks_stat, ks_pval = stats.ks_2samp(real_feat, synth_feat)

        # Jensen-Shannon divergence
        # Create histograms
        hist_real, bins = np.histogram(real_feat, bins=50, density=True)
        hist_synth, _ = np.histogram(synth_feat, bins=bins, density=True)

        # Normalize to probability distributions
        hist_real = hist_real / hist_real.sum()
        hist_synth = hist_synth / hist_synth.sum()

        # JS divergence
        js_div = jensenshannon(hist_real, hist_synth)

        # Interpretation
        ks_status = "✅ Similar" if ks_pval > 0.05 else "⚠️ Different"
        js_status = "✅ Similar" if js_div < 0.1 else "⚠️ Different"

        results.append({
            'feature': name,
            'ks_statistic': ks_stat,
            'ks_pvalue': ks_pval,
            'ks_result': ks_status,
            'js_divergence': js_div,
            'js_result': js_status
        })

        print(f"{name:<20} KS: {ks_stat:.4f} (p={ks_pval:.4f}) {ks_status}  JS: {js_div:.4f} {js_status}")

    df = pd.DataFrame(results)
    df.to_csv(f'{output_path}/test2_distribution_similarity.csv', index=False)

    similar_ks = sum(1 for r in results if r['ks_pvalue'] > 0.05)
    similar_js = sum(1 for r in results if r['js_divergence'] < 0.1)
    print(f"\n📊 Summary:")
    print(f"   KS test: {similar_ks}/{len(results)} features have similar distributions (p>0.05)")
    print(f"   JS divergence: {similar_js}/{len(results)} features are similar (JS<0.1)")
    print(f"✅ Saved: test2_distribution_similarity.csv")

    return df


def test_3_correlation_matrix(real, synthetic, output_path):
    """Test 3: Compare correlation matrices"""

    print("\n" + "="*70)
    print("TEST 3: CORRELATION MATRIX COMPARISON")
    print("="*70)

    feature_names = ['acc_z', 'acc_y', 'acc_x', 'gyro_z', 'gyro_y', 'gyro_x',
                     'mag_z', 'mag_y', 'mag_x', 'orient_qz', 'orient_qy', 'orient_qx',
                     'orient_qw', 'orient_roll', 'orient_pitch', 'orient_yaw',
                     'grav_z', 'grav_y', 'grav_x']

    # Sample data for correlation (use first 1000 timesteps)
    real_sample = real[:50].reshape(-1, 19)
    synth_sample = synthetic[:50].reshape(-1, 19)

    # Compute correlation matrices
    real_corr = np.corrcoef(real_sample.T)
    synth_corr = np.corrcoef(synth_sample.T)

    # Compute difference
    corr_diff = np.abs(real_corr - synth_corr)
    avg_diff = corr_diff.mean()
    max_diff = corr_diff.max()

    print(f"Average correlation difference: {avg_diff:.4f}")
    print(f"Maximum correlation difference: {max_diff:.4f}")

    status = "✅ GOOD" if avg_diff < 0.15 else "⚠️ CHECK"
    print(f"Status: {status} (target: avg < 0.15)")

    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # Real correlation
    sns.heatmap(real_corr, ax=axes[0], cmap='coolwarm', center=0, vmin=-1, vmax=1,
                xticklabels=feature_names, yticklabels=feature_names, cbar_kws={'label': 'Correlation'})
    axes[0].set_title('Real Data Correlation', fontweight='bold', fontsize=12)

    # Synthetic correlation
    sns.heatmap(synth_corr, ax=axes[1], cmap='coolwarm', center=0, vmin=-1, vmax=1,
                xticklabels=feature_names, yticklabels=feature_names, cbar_kws={'label': 'Correlation'})
    axes[1].set_title('Synthetic Data Correlation', fontweight='bold', fontsize=12)

    # Difference
    sns.heatmap(corr_diff, ax=axes[2], cmap='Reds', vmin=0, vmax=0.5,
                xticklabels=feature_names, yticklabels=feature_names, cbar_kws={'label': 'Absolute Difference'})
    axes[2].set_title(f'Correlation Difference (Avg: {avg_diff:.3f})', fontweight='bold', fontsize=12)

    plt.tight_layout()
    plt.savefig(f'{output_path}/test3_correlation_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()

    # Save matrices
    pd.DataFrame(real_corr, columns=feature_names, index=feature_names).to_csv(
        f'{output_path}/test3_real_correlation.csv')
    pd.DataFrame(synth_corr, columns=feature_names, index=feature_names).to_csv(
        f'{output_path}/test3_synthetic_correlation.csv')

    print(f"✅ Saved: test3_correlation_comparison.png")
    print(f"✅ Saved: correlation matrices as CSV")

    return avg_diff, max_diff


def test_4_frequency_domain(real, synthetic, output_path):
    """Test 4: Compare frequency domain characteristics (FFT)"""

    print("\n" + "="*70)
    print("TEST 4: FREQUENCY DOMAIN ANALYSIS")
    print("="*70)

    # Focus on motion sensors
    sensor_groups = [
        ('Accelerometer', [0, 1, 2], ['acc_z', 'acc_y', 'acc_x']),
        ('Gyroscope', [3, 4, 5], ['gyro_z', 'gyro_y', 'gyro_x']),
        ('Magnetometer', [6, 7, 8], ['mag_z', 'mag_y', 'mag_x'])
    ]

    fig, axes = plt.subplots(3, 3, figsize=(18, 12))

    for row, (sensor_name, indices, labels) in enumerate(sensor_groups):
        for col, (idx, label) in enumerate(zip(indices, labels)):
            ax = axes[row, col]

            # Get data
            real_seq = real[0, :, idx]
            synth_seq = synthetic[0, :, idx]

            # Compute FFT
            real_fft = fft(real_seq)
            synth_fft = fft(synth_seq)
            freqs = fftfreq(len(real_seq), d=0.01)  # 100 Hz sampling

            # Plot positive frequencies only
            mask = freqs > 0
            ax.plot(freqs[mask], np.abs(real_fft[mask]), label='Real', alpha=0.7, linewidth=2)
            ax.plot(freqs[mask], np.abs(synth_fft[mask]), label='Synthetic', alpha=0.7, linewidth=2)

            ax.set_xlabel('Frequency (Hz)', fontweight='bold')
            ax.set_ylabel('Magnitude', fontweight='bold')
            ax.set_title(f'{sensor_name} - {label}', fontweight='bold')
            ax.legend()
            ax.grid(True, alpha=0.3)
            ax.set_xlim([0, 10])  # Focus on 0-10 Hz

    plt.suptitle('Frequency Domain Comparison (First Sequence)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_path}/test4_frequency_domain.png', dpi=150, bbox_inches='tight')
    plt.close()

    print(f"✅ Saved: test4_frequency_domain.png")
    print(f"Check if synthetic data has similar frequency peaks as real data")

    return None


def test_5_temporal_continuity(real, synthetic, output_path):
    """Test 5: Check temporal smoothness and continuity"""

    print("\n" + "="*70)
    print("TEST 5: TEMPORAL CONTINUITY")
    print("="*70)

    feature_names = ['acc_z', 'acc_y', 'acc_x', 'gyro_z', 'gyro_y', 'gyro_x',
                     'mag_z', 'mag_y', 'mag_x', 'orient_qz', 'orient_qy', 'orient_qx',
                     'orient_qw', 'orient_roll', 'orient_pitch', 'orient_yaw',
                     'grav_z', 'grav_y', 'grav_z']

    results = []
    for i, name in enumerate(feature_names[:19]):
        # Compute first-order differences (velocity)
        real_diff = np.diff(real[:, :, i], axis=1)
        synth_diff = np.diff(synthetic[:, :, i], axis=1)

        # Average absolute difference (smoothness metric)
        real_smoothness = np.abs(real_diff).mean()
        synth_smoothness = np.abs(synth_diff).mean()

        # Standard deviation of differences (variability)
        real_var = real_diff.std()
        synth_var = synth_diff.std()

        results.append({
            'feature': name,
            'real_smoothness': real_smoothness,
            'synth_smoothness': synth_smoothness,
            'smoothness_ratio': synth_smoothness / (real_smoothness + 1e-10),
            'real_variability': real_var,
            'synth_variability': synth_var,
            'variability_ratio': synth_var / (real_var + 1e-10)
        })

        ratio = synth_smoothness / (real_smoothness + 1e-10)
        status = "✅ GOOD" if 0.5 < ratio < 2.0 else "⚠️ CHECK"
        print(f"{name:<20} Smoothness ratio: {ratio:.3f} {status}")

    df = pd.DataFrame(results)
    df.to_csv(f'{output_path}/test5_temporal_continuity.csv', index=False)

    good_count = sum(1 for r in results if 0.5 < r['smoothness_ratio'] < 2.0)
    print(f"\n📊 Summary: {good_count}/{len(results)} features have similar temporal smoothness")
    print(f"✅ Saved: test5_temporal_continuity.csv")

    return df


def test_6_visual_comparison(real, synthetic, output_path):
    """Test 6: Visual side-by-side comparison"""

    print("\n" + "="*70)
    print("TEST 6: VISUAL COMPARISON")
    print("="*70)

    # Plot 5 random sequences from each
    n_samples = 5

    sensor_groups = [
        ('Accelerometer', [0, 1, 2], ['acc_z', 'acc_y', 'acc_x']),
        ('Gyroscope', [3, 4, 5], ['gyro_z', 'gyro_y', 'gyro_x'])
    ]

    for sensor_name, indices, labels in sensor_groups:
        fig, axes = plt.subplots(n_samples, 2, figsize=(16, 12), sharex=True)

        real_indices = np.random.choice(len(real), n_samples, replace=False)
        synth_indices = np.random.choice(len(synthetic), n_samples, replace=False)

        for i in range(n_samples):
            # Real data
            ax = axes[i, 0]
            for idx, label in zip(indices, labels):
                ax.plot(real[real_indices[i], :, idx], label=label, alpha=0.7)
            ax.set_ylabel(f'Seq {i+1}', fontweight='bold')
            ax.grid(True, alpha=0.3)
            if i == 0:
                ax.legend(loc='upper right', ncol=3)
                ax.set_title('REAL DATA', fontweight='bold', fontsize=12)

            # Synthetic data
            ax = axes[i, 1]
            for idx, label in zip(indices, labels):
                ax.plot(synthetic[synth_indices[i], :, idx], label=label, alpha=0.7)
            ax.set_ylabel(f'Seq {i+1}', fontweight='bold')
            ax.grid(True, alpha=0.3)
            if i == 0:
                ax.legend(loc='upper right', ncol=3)
                ax.set_title('SYNTHETIC DATA', fontweight='bold', fontsize=12)

        axes[-1, 0].set_xlabel('Timesteps', fontweight='bold')
        axes[-1, 1].set_xlabel('Timesteps', fontweight='bold')

        plt.suptitle(f'{sensor_name} - Real vs Synthetic Comparison', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'{output_path}/test6_visual_{sensor_name.lower()}.png', dpi=150, bbox_inches='tight')
        plt.close()

    print(f"✅ Saved: test6_visual_accelerometer.png")
    print(f"✅ Saved: test6_visual_gyroscope.png")
    print(f"Visually inspect: synthetic should look similar to real (smooth, realistic patterns)")

    return None


def test_7_pca_comparison(real, synthetic, output_path):
    """Test 7: Compare PCA projections"""

    print("\n" + "="*70)
    print("TEST 7: PCA PROJECTION COMPARISON")
    print("="*70)

    from sklearn.decomposition import PCA

    # Flatten sequences
    real_flat = real.reshape(-1, 19)
    synth_flat = synthetic.reshape(-1, 19)

    # Fit PCA on real data
    pca = PCA(n_components=2)
    real_pca = pca.fit_transform(real_flat)
    synth_pca = pca.transform(synth_flat)

    # Sample for plotting (too many points otherwise)
    sample_size = 5000
    real_sample_idx = np.random.choice(len(real_pca), min(sample_size, len(real_pca)), replace=False)
    synth_sample_idx = np.random.choice(len(synth_pca), min(sample_size, len(synth_pca)), replace=False)

    # Plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.scatter(real_pca[real_sample_idx, 0], real_pca[real_sample_idx, 1],
               alpha=0.3, s=10, label='Real', c='blue')
    ax.scatter(synth_pca[synth_sample_idx, 0], synth_pca[synth_sample_idx, 1],
               alpha=0.3, s=10, label='Synthetic', c='red')
    ax.set_xlabel('PC1', fontweight='bold')
    ax.set_ylabel('PC2', fontweight='bold')
    ax.set_title('PCA Projection: Real vs Synthetic', fontweight='bold', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{output_path}/test7_pca_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()

    print(f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}, PC2={pca.explained_variance_ratio_[1]:.3f}")
    print(f"✅ Saved: test7_pca_comparison.png")
    print(f"Check if synthetic data (red) overlaps with real data (blue) in PCA space")

    return None


def generate_validation_report(output_path):
    """Generate final validation report"""

    print("\n" + "="*70)
    print("GENERATING VALIDATION REPORT")
    print("="*70)

    report = f"""
# Synthetic Data Validation Report

## Overview
This report summarizes the validation of synthetic sensor data against real data.

## Tests Performed

### Test 1: Basic Statistics ✅
- File: test1_basic_statistics.csv
- Checks: Mean, Std, Min, Max for each feature
- Target: Mean difference < 20%, Std difference < 30%

### Test 2: Distribution Similarity ✅
- File: test2_distribution_similarity.csv
- Checks: KS test (p-value) and Jensen-Shannon divergence
- Target: KS p-value > 0.05, JS divergence < 0.1

### Test 3: Correlation Matrix ✅
- File: test3_correlation_comparison.png
- Checks: Correlation structure preservation
- Target: Average difference < 0.15

### Test 4: Frequency Domain ✅
- File: test4_frequency_domain.png
- Checks: FFT comparison for motion sensors
- Visual inspection: Similar frequency peaks

### Test 5: Temporal Continuity ✅
- File: test5_temporal_continuity.csv
- Checks: Smoothness and temporal variability
- Target: Smoothness ratio between 0.5-2.0

### Test 6: Visual Comparison ✅
- Files: test6_visual_*.png
- Checks: Visual similarity of sequences
- Visual inspection: Realistic patterns

### Test 7: PCA Projection ✅
- File: test7_pca_comparison.png
- Checks: Data distribution in reduced space
- Visual inspection: Overlap in PCA space

## How to Interpret Results

### ✅ GOOD Quality Indicators:
- Mean/Std differences < 20-30%
- KS test p-value > 0.05
- JS divergence < 0.1
- Correlation difference < 0.15
- Visual similarity in plots
- Synthetic overlaps with real in PCA

### ⚠️ CHECK Indicators:
- Mean/Std differences > 30%
- KS test p-value < 0.05
- JS divergence > 0.2
- Correlation difference > 0.25
- Synthetic looks too smooth/noisy
- Poor overlap in PCA

### ❌ POOR Quality Indicators:
- Mean/Std differences > 50%
- JS divergence > 0.5
- Correlation structure very different
- Unrealistic visual patterns
- No overlap in PCA space

## Next Steps

If validation passes (mostly ✅):
→ Use synthetic data for anti-emulation detection
→ Generate more samples if needed

If validation has issues (some ⚠️):
→ Review specific features with problems
→ Consider retraining with adjusted hyperparameters
→ Check if real data needs more preprocessing

If validation fails (mostly ❌):
→ Retrain TimeGAN with different architecture
→ Check data preprocessing pipeline
→ Verify scaler is correctly applied
"""

    with open(f'{output_path}/VALIDATION_REPORT.md', 'w') as f:
        f.write(report)

    print(f"✅ Saved: VALIDATION_REPORT.md")
    print(f"\nAll validation outputs saved to: {output_path}/")

    return None


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":

    # ========== CONFIGURATION ==========

    # Paths to data
    REAL_DATA_PATH = '/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/train_sequences_19features.npy'
    SYNTHETIC_DATA_PATH = '/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/synthetic_output/synthetic_denormalized_383.npy'
    SCALER_PATH = '/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/scaler_19features.pkl'

    # Output directory
    OUTPUT_PATH = '/content/drive/MyDrive/CS5103_IITH_PMA_project/Cycling/validation_results'

    # ========== END CONFIGURATION ==========

    # Create output directory
    os.makedirs(OUTPUT_PATH, exist_ok=True)

    # Load data
    real_norm, real_denorm, synthetic_denorm = load_data(
        REAL_DATA_PATH, SYNTHETIC_DATA_PATH, SCALER_PATH
    )

    # Run all tests
    test_1_basic_statistics(real_denorm, synthetic_denorm, OUTPUT_PATH)
    test_2_distribution_similarity(real_denorm, synthetic_denorm, OUTPUT_PATH)
    test_3_correlation_matrix(real_denorm, synthetic_denorm, OUTPUT_PATH)
    test_4_frequency_domain(real_denorm, synthetic_denorm, OUTPUT_PATH)
    test_5_temporal_continuity(real_denorm, synthetic_denorm, OUTPUT_PATH)
    test_6_visual_comparison(real_denorm, synthetic_denorm, OUTPUT_PATH)
    test_7_pca_comparison(real_denorm, synthetic_denorm, OUTPUT_PATH)

    # Generate report
    generate_validation_report(OUTPUT_PATH)

    print("\n" + "="*70)
    print("🎉 VALIDATION COMPLETE!")
    print("="*70)
    print(f"\nAll results saved to: {OUTPUT_PATH}/")
    print("\nReview:")
    print("  1. CSV files for quantitative metrics")
    print("  2. PNG files for visual inspection")
    print("  3. VALIDATION_REPORT.md for interpretation guide")

LOADING DATA
✅ Real data (normalized): (383, 100, 19)
✅ Synthetic data (denormalized): (383, 100, 19)
✅ Real data (denormalized): (383, 100, 19)

TEST 1: BASIC STATISTICS COMPARISON
acc_z                Mean:     0.08 vs    -2.25 (2776.8%)  ⚠️ CHECK
acc_y                Mean:    -0.04 vs    -1.20 (2629.5%)  ⚠️ CHECK
acc_x                Mean:    -0.02 vs     0.01 (123.3%)  ⚠️ CHECK
gyro_z               Mean:     0.01 vs     0.71 (12834.3%)  ⚠️ CHECK
gyro_y               Mean:    -0.02 vs     0.13 (932.3%)  ⚠️ CHECK
gyro_x               Mean:    -0.00 vs     0.24 (6677.9%)  ⚠️ CHECK
mag_z                Mean:     8.58 vs   -10.58 (223.3%)  ⚠️ CHECK
mag_y                Mean:     2.02 vs     4.69 (132.2%)  ⚠️ CHECK
mag_x                Mean:   -23.23 vs   -16.82 ( 27.6%)  ⚠️ CHECK
orient_qz            Mean:    -0.11 vs    -0.02 ( 85.7%)  ⚠️ CHECK
orient_qy            Mean:     0.06 vs    -0.22 (459.5%)  ⚠️ CHECK
orient_qx            Mean:    -0.47 vs    -0.30 ( 35.9%)  ⚠️ CHECK
orient_qw

In [ ]:
"""
timegan_pytorch_run.py
Ready-to-run TimeGAN (PyTorch) script for Python 3.12.

Usage:
    python timegan_pytorch_run.py

Ensure dependencies are installed:
    pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118  # or CPU build
    pip install numpy pandas scikit-learn matplotlib tqdm

Adjust HYPERPARAMETERS below as needed.
"""

import os
import math
import random
import numpy as np
import pandas as pd
from tqdm import trange
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# ----------------------------
# HYPERPARAMETERS
# ----------------------------
DATA_PATH = "/content/drive/MyDrive/Datasets/running_sensors_100hz_trimmed.csv"  # trimmed file
OUTPUT_SYN_PATH = "/content/drive/MyDrive/Datasets/synthetic_timegan_samples.npy"

SEQ_LEN = 100             # sequence length (window size)
FEATURE_COLUMNS = None    # None -> use all numeric columns except time column if present
TIME_COLUMN_NAME = "time_seconds"  # if your CSV has a time column; will be excluded from features
BATCH_SIZE = 64
HIDDEN_DIM = 64
EMB_DIM = 32
NOISE_DIM = 32
NUM_LAYERS = 2
LEARNING_RATE = 1e-3
EPOCHS = 500               # higher for better results
G_STEPS = 1
D_STEPS = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# ----------------------------
# DATASET / PREPROCESS
# ----------------------------
class TimeSeriesDataset(Dataset):
    def __init__(self, sequences):
        # sequences: numpy array (n_samples, seq_len, n_features)
        self.sequences = sequences.astype(np.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]

def load_and_preprocess(path, seq_len=SEQ_LEN, time_col=TIME_COLUMN_NAME, feature_cols=FEATURE_COLUMNS):
    df = pd.read_csv(path)
    # Drop non-numeric columns except feature columns if user specified
    if feature_cols is None:
        # exclude time column if exists
        cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if time_col in cols:
            cols.remove(time_col)
    else:
        cols = feature_cols

    data = df[cols].values.astype(np.float32)
    # Scale features to [0,1]
    scaler = MinMaxScaler()
    data = scaler.fit_transform(data)

    # Sliding windows to create sequences
    sequences = []
    n_rows = data.shape[0]
    for start in range(0, n_rows - seq_len + 1, seq_len):  # non-overlapping windows
        seq = data[start:start + seq_len]
        if seq.shape[0] == seq_len:
            sequences.append(seq)
    sequences = np.stack(sequences)  # (n_samples, seq_len, n_features)
    return sequences, scaler, cols

# ----------------------------
# MODEL COMPONENTS
# ----------------------------
# Simple RNN-based modules for embedding, recovery, generator, supervisor, discriminator

class EmbeddedRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, emb_dim, n_layers=1):
        super().__init__()
        self.rnn = nn.GRU(input_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, emb_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        # x: (B, T, input_dim)
        out, _ = self.rnn(x)
        # map each time-step to embedding
        emb = self.relu(self.fc(out))
        return emb  # (B, T, emb_dim)

class RecoveryRNN(nn.Module):
    def __init__(self, emb_dim, hidden_dim, output_dim, n_layers=1):
        super().__init__()
        self.rnn = nn.GRU(emb_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, emb):
        out, _ = self.rnn(emb)
        rec = self.fc(out)
        return rec  # (B, T, output_dim)

class GeneratorRNN(nn.Module):
    def __init__(self, noise_dim, hidden_dim, emb_dim, n_layers=1):
        super().__init__()
        self.rnn = nn.GRU(noise_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, emb_dim)
        self.relu = nn.ReLU()

    def forward(self, z):
        out, _ = self.rnn(z)
        emb = self.relu(self.fc(out))
        return emb

class SupervisorRNN(nn.Module):
    def __init__(self, emb_dim, hidden_dim, n_layers=1):
        super().__init__()
        self.rnn = nn.GRU(emb_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, emb_dim)
        self.relu = nn.ReLU()

    def forward(self, emb):
        out, _ = self.rnn(emb)
        sup = self.relu(self.fc(out))
        return sup

class DiscriminatorRNN(nn.Module):
    def __init__(self, emb_dim, hidden_dim, n_layers=1):
        super().__init__()
        self.rnn = nn.GRU(emb_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, emb):
        out, _ = self.rnn(emb)
        # use last time step output for classification
        last = out[:, -1, :]
        y = self.sigmoid(self.fc(last))
        return y.squeeze(-1)

# ----------------------------
# TIMEGAN WRAPPER
# ----------------------------
class TimeGAN:
    def __init__(self, feature_dim, seq_len, hidden_dim=HIDDEN_DIM, emb_dim=EMB_DIM, noise_dim=NOISE_DIM, n_layers=NUM_LAYERS, device=DEVICE):
        self.feature_dim = feature_dim
        self.seq_len = seq_len
        self.hidden_dim = hidden_dim
        self.emb_dim = emb_dim
        self.noise_dim = noise_dim
        self.n_layers = n_layers
        self.device = device

        # Modules
        self.emb_net = EmbeddedRNN(feature_dim, hidden_dim, emb_dim, n_layers).to(device)
        self.rec_net = RecoveryRNN(emb_dim, hidden_dim, feature_dim, n_layers).to(device)
        self.gen_net = GeneratorRNN(noise_dim, hidden_dim, emb_dim, n_layers).to(device)
        self.sup_net = SupervisorRNN(emb_dim, hidden_dim, n_layers).to(device)
        self.disc_net = DiscriminatorRNN(emb_dim, hidden_dim, n_layers).to(device)

        # Optimizers
        self.opt_e = torch.optim.Adam(list(self.emb_net.parameters()) + list(self.rec_net.parameters()), lr=LEARNING_RATE)
        self.opt_g = torch.optim.Adam(list(self.gen_net.parameters()) + list(self.sup_net.parameters()), lr=LEARNING_RATE)
        self.opt_d = torch.optim.Adam(self.disc_net.parameters(), lr=LEARNING_RATE)

        # Losses
        self.mse = nn.MSELoss()
        self.bce = nn.BCELoss()

    def embed(self, x):
        return self.emb_net(x)

    def recover(self, emb):
        return self.rec_net(emb)

    def supervise(self, emb):
        return self.sup_net(emb)

    def generate(self, z):
        return self.gen_net(z)

    def discriminate(self, emb):
        return self.disc_net(emb)

    def train_step_embed_recover(self, X):
        # X: (B, T, feature_dim)
        self.opt_e.zero_grad()
        emb = self.embed(X)
        X_tilde = self.recover(emb)
        loss = self.mse(X_tilde, X)
        loss.backward()
        self.opt_e.step()
        return loss.item()

    def train_step_supervised(self, X):
        # Train supervisor to minimize supervised loss between embedding and its next-step prediction
        self.opt_g.zero_grad()
        emb = self.embed(X).detach()  # detach to not update embed/net
        H = emb
        H_hat = self.supervise(H)
        loss = self.mse(H_hat[:, :-1, :], H[:, 1:, :])  # predict next step embedding
        loss.backward()
        self.opt_g.step()
        return loss.item()

    def train_step_adversarial(self, X):
        # Adversarial training: generator (with supervisor) vs discriminator
        # Prepare real embeddings
        emb_real = self.embed(X).detach()
        # Generator produce fake embeddings from noise
        B = X.shape[0]
        z = torch.randn(B, self.seq_len, self.noise_dim, device=self.device)
        emb_fake = self.generate(z)
        emb_fake_sup = self.supervise(emb_fake)

        # Discriminator step
        self.opt_d.zero_grad()
        y_real = self.discriminate(emb_real)
        y_fake = self.discriminate(emb_fake_sup.detach())
        real_labels = torch.ones_like(y_real, device=self.device)
        fake_labels = torch.zeros_like(y_fake, device=self.device)
        d_loss = (self.bce(y_real, real_labels) + self.bce(y_fake, fake_labels)) / 2
        d_loss.backward()
        self.opt_d.step()

        # Generator step
        self.opt_g.zero_grad()
        y_fake_for_g = self.discriminate(emb_fake_sup)
        g_loss_adv = self.bce(y_fake_for_g, real_labels)
        # Two more losses: supervised and reconstruction (via recovery network)
        X_tilde = self.recover(emb_fake_sup)
        g_loss_rec = self.mse(X_tilde, X)
        # Total generator loss (weighted)
        g_loss = g_loss_adv + 100 * g_loss_rec  # weight reconstruction strongly
        g_loss.backward()
        self.opt_g.step()

        return d_loss.item(), g_loss.item(), g_loss_adv.item(), g_loss_rec.item()

    def sample(self, n_samples):
        # sample n_samples synthetic sequences
        self.eval_mode()
        with torch.no_grad():
            z = torch.randn(n_samples, self.seq_len, self.noise_dim, device=self.device)
            emb_fake = self.generate(z)
            emb_fake_sup = self.supervise(emb_fake)
            X_tilde = self.recover(emb_fake_sup)
            return X_tilde.cpu().numpy()

    def eval_mode(self):
        self.emb_net.eval()
        self.rec_net.eval()
        self.gen_net.eval()
        self.sup_net.eval()
        self.disc_net.eval()

    def train_model(self, train_loader, epochs=EPOCHS):
        # Simple training loop combining stages: embed/recover pretrain, supervisor pretrain, adversarial train
        # Stage 1: embed-recover pretrain
        print("Stage 1: pretraining embed & recover")
        for epoch in range(5):
            losses = []
            for X in train_loader:
                X = X.to(self.device)
                loss = self.train_step_embed_recover(X)
                losses.append(loss)
            print(f"ER pretrain epoch {epoch+1} loss: {np.mean(losses):.6f}")

        # Stage 2: supervise pretrain
        print("Stage 2: pretrain supervisor")
        for epoch in range(5):
            losses = []
            for X in train_loader:
                X = X.to(self.device)
                loss = self.train_step_supervised(X)
                losses.append(loss)
            print(f"Supervisor pretrain epoch {epoch+1} loss: {np.mean(losses):.6f}")

        # Stage 3: joint adversarial training
        print("Stage 3: adversarial training")
        for epoch in range(epochs):
            d_losses = []
            g_losses = []
            rec_losses = []
            adv_losses = []
            for X in train_loader:
                X = X.to(self.device)
                d_loss, g_loss, g_adv, g_rec = self.train_step_adversarial(X)
                d_losses.append(d_loss)
                g_losses.append(g_loss)
                adv_losses.append(g_adv)
                rec_losses.append(g_rec)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"Epoch {epoch+1}/{epochs} | D_loss: {np.mean(d_losses):.6f} | G_loss: {np.mean(g_losses):.6f} | G_adv: {np.mean(adv_losses):.6f} | G_rec: {np.mean(rec_losses):.6f}")

# ----------------------------
# TRAINING ENTRYPOINT
# ----------------------------
def main():
    print("Loading and preprocessing data...")
    sequences, scaler, feature_cols = load_and_preprocess(DATA_PATH, seq_len=SEQ_LEN)
    print("Sequences shape:", sequences.shape)  # (n_samples, seq_len, n_features)

    dataset = TimeSeriesDataset(sequences)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    n_features = sequences.shape[2]
    model = TimeGAN(feature_dim=n_features, seq_len=SEQ_LEN, hidden_dim=HIDDEN_DIM, emb_dim=EMB_DIM, noise_dim=NOISE_DIM, n_layers=NUM_LAYERS, device=DEVICE)
    model.train_model(loader, epochs=EPOCHS)

    print("Sampling synthetic sequences...")
    synthetic = model.sample(500)  # (n_samples, seq_len, n_features)
    np.save(OUTPUT_SYN_PATH, synthetic)
    print("Saved synthetic samples to:", OUTPUT_SYN_PATH)

if __name__ == "__main__":
    main()
